In [1]:
import nest_asyncio
nest_asyncio.apply()  # Cho phép nest event loop trong Jupyter

import sys
sys.path.insert(0, "/home/duoc/workflow/api")  # thư mục chứa folder app/
print("Cấu hình sys.path thành công! Thư mục hiện tại:")
import os
os.chdir("/home/duoc/workflow/api")  # Thêm dòng này TRƯỚC khi import app

Cấu hình sys.path thành công! Thư mục hiện tại:


1. Người dùng muốn gì?

→ Node 1

2. Thị trường đang viết gì để rank?

→ Node 2

3. Mình phải viết gì để thắng?

→ Node 3

In [2]:
import asyncio
from urllib.parse import quote
from bs4 import BeautifulSoup
import nodriver as uc

QUERY = "nhà hàng quán nhậu hải sản Đà Nẵng"


# =========================
# CLEAN
# =========================
def clean(text: str):
    if not text:
        return None

    text = " ".join(text.split())

    BAD = [
        "facebook",
        "tripadvisor",
        "đăng nhập",
        "chính sách",
        "hỗ trợ",
        "video",
        "youtube",
        "maps",
        "translate",
        "google",
        "javascript",
        "cookie"
    ]

    if (
        len(text) < 5
        or len(text) > 160
        or any(x in text.lower() for x in BAD)
    ):
        return None

    return text


def is_valid_url(url: str):
    if not url or not url.startswith("http"):
        return False

    BAD_URL = [
        "google.com/search",
        "googleusercontent",
        "webcache",
        "/settings",
        "accounts.google"
    ]

    return not any(x in url for x in BAD_URL)


# =========================
# NODE 1 ENGINE
# =========================
async def node1_serp():

    browser = await uc.start(headless=False)

    try:
        page = await browser.get(
            f"https://www.google.com/search?q={quote(QUERY)}&hl=vi"
        )

        await asyncio.sleep(3)

        soup = BeautifulSoup(await page.get_content(), "html.parser")

        # =========================
        # 1. TOP URLS (Organic)
        # =========================
        top_urls = []
        seen = set()

        for a in soup.select("a"):
            h3 = a.select_one("h3")
            if not h3:
                continue

            url = a.get("href")
            title = clean(h3.get_text(" ", strip=True))

            if not title or not is_valid_url(url):
                continue

            key = url.split("#")[0]
            if key in seen:
                continue

            seen.add(key)

            domain = url.split("/")[2] if "://" in url else ""

            top_urls.append({
                "position": len(top_urls) + 1,
                "title": title,
                "url": url,
                "domain": domain,
                "type": "organic"
            })

        # =========================
        # 2. PEOPLE ALSO ASK (PAA)
        # =========================
        paa = []

        for el in soup.select('div[jsname="N760b"]'):
            q = clean(el.get_text(" ", strip=True))
            if q and "?" in q:
                paa.append({
                    "question": q,
                    "intent": "expansion"
                })

        paa = paa[:5]

        # =========================
        # 3. RELATED SEARCHES
        # =========================
        related = []

        for a in soup.select("a"):
            text = clean(a.get_text(" ", strip=True))

            if not text:
                continue

            if any(k in text.lower() for k in ["quán", "hải sản", "đà nẵng"]):
                if len(text.split()) <= 10:
                    related.append(text)

        related = list(dict.fromkeys(related))[:10]

        # =========================
        # 4. SNIPPETS (clean SERP preview)
        # =========================
        snippets = []

        for block in soup.select("div.g"):
            snippet_el = block.select_one("span")

            if not snippet_el:
                continue

            text = clean(snippet_el.get_text(" ", strip=True))

            if text and 60 < len(text) < 300:
                snippets.append({
                    "snippet": text[:200]
                })

        snippets = snippets[:5]

        # =========================
        # 5. KEYWORD CLUSTER
        # =========================
        keyword_cluster = [
            "quán hải sản ngon Đà Nẵng",
            "hải sản giá rẻ",
            "gần biển Mỹ Khê",
            "Sơn Trà",
            "quán nhậu Đà Nẵng",
            "buffet hải sản"
        ]

        # =========================
        # 6. CONTENT ANGLE
        # =========================
        content_angle = [
            "review",
            "top list",
            "trải nghiệm",
            "giá rẻ",
            "du lịch ăn uống"
        ]

        # =========================
        # 7. INTENT LAYER
        # =========================
        intent = [
            "ăn nhậu",
            "du lịch",
            "gia đình",
            "buffet",
            "hải sản tươi sống"
        ]

        # =========================
        # 8. COMPETITOR PATTERN
        # =========================
        competitor_pattern = [
            "Pasgo",
            "Tripadvisor",
            "Blog du lịch SEO",
            "Facebook group",
            "Local listing / directory"
        ]

        # =========================
        # FINAL OUTPUT
        # =========================
        return {
            "query": QUERY,
            "top_urls": top_urls[:10],
            "people_also_ask": paa,
            "related_searches": related,
            "snippets": snippets,
            "keyword_cluster": keyword_cluster,
            "content_angle": content_angle,
            "intent": intent,
            "competitor_pattern": competitor_pattern
        }

    finally:
        if browser:
            browser.stop()


# =========================
# JUPYTER RUN
# =========================
node1 = await node1_serp()
from pprint import pprint
pprint(node1)

{'competitor_pattern': ['Pasgo',
                        'Tripadvisor',
                        'Blog du lịch SEO',
                        'Facebook group',
                        'Local listing / directory'],
 'content_angle': ['review',
                   'top list',
                   'trải nghiệm',
                   'giá rẻ',
                   'du lịch ăn uống'],
 'intent': ['ăn nhậu', 'du lịch', 'gia đình', 'buffet', 'hải sản tươi sống'],
 'keyword_cluster': ['quán hải sản ngon Đà Nẵng',
                     'hải sản giá rẻ',
                     'gần biển Mỹ Khê',
                     'Sơn Trà',
                     'quán nhậu Đà Nẵng',
                     'buffet hải sản'],
 'people_also_ask': [],
 'query': 'nhà hàng quán nhậu hải sản Đà Nẵng',
 'related_searches': ['Hải sản Thời Cổ',
                      'Hải sản Năm Đảnh',
                      'Hải sản Mộc quán',
                      'Hải Sản Gành Bàng',
                      'Quán Eo Biển',
                      'Hải 

In [3]:
import json
import re
from groq import Groq
from app.llm_clients import call_groq


from app.llm_clients import call_groq


def node2_llm(node1):

    payload = {
        "query": node1.get("query", ""),
        "top_urls": [x.get("title") for x in node1.get("top_urls", [])],
        "related_searches": node1.get("related_searches", []),
        "snippets": node1.get("snippets", [])
    }

    prompt = f"""
You are a SERP understanding layer.

You MUST output ONLY in this structure:

INTENT:
- ...

EXPANSION:
- ...

CONTENT SIGNALS:
- price:
- quality:
- format:
- location:

SERP TYPE:
- one or more labels

ENTITIES:
- only real named entities from input

MARKET INSIGHT:
- one sentence only

CRITICAL RULES:
- DO NOT repeat this instruction
- DO NOT output RULES section
- DO NOT add any extra headings
- ONLY use the sections above

INPUT:
{payload}
"""

    return call_groq(prompt)

# =========================
# 3. RUN
# =========================
node2 = node2_llm(node1)

print("\n===== RAW LLM OUTPUT =====\n")
print(node2)




===== RAW LLM OUTPUT =====

INTENT:
- Tìm kiếm nhà hàng quán nhậu hải sản tại Đà Nẵng

EXPANSION:
- Tìm kiếm thông tin về quán hải sản tại Đà Nẵng
- Tìm kiếm đánh giá và review về quán hải sản tại Đà Nẵng
- Tìm kiếm quán hải sản giá rẻ và chất lượng tại Đà Nẵng

CONTENT SIGNALS:
- price: giá rẻ, giá bình dân
- quality: tươi ngon, chất lượng
- format: quán ăn, nhà hàng
- location: Đà Nẵng, Mỹ Khê

SERP TYPE:
- informational
- navigational
- review

ENTITIES:
- Đà Nẵng
- Mỹ Khê

MARKET INSIGHT:
- Người dùng đang tìm kiếm thông tin về quán hải sản tại Đà Nẵng với mức giá hợp lý và chất lượng tốt.


In [3]:
import asyncio
from collections import Counter
from bs4 import BeautifulSoup
import nodriver as uc


# =========================
# CLEAN TEXT
# =========================
def clean_text(text: str):
    return " ".join(text.split()).strip()


# =========================
# DETECT SIMPLE STRUCTURE (GENERIC)
# =========================
def detect_structure(text: str):
    t = text.lower()

    score = {
        "listicle": t.count("1.") + t.count("2.") + t.count("top") + t.count("danh sách"),
        "review": t.count("trải nghiệm") + t.count("đánh giá"),
        "guide": t.count("hướng dẫn") + t.count("cách"),
        "informational": t.count("là gì") + t.count("khái niệm"),
    }

    return max(score, key=score.get)


# =========================
# EXTRACT CONTENT SIGNALS (NO DOMAIN KNOWLEDGE)
# =========================
def extract_signals(text: str):

    t = text.lower()

    signals = {
        "price": sum([t.count(x) for x in ["giá", "rẻ", "đắt", "chi phí"]]),
        "quality": sum([t.count(x) for x in ["ngon", "tốt", "chất lượng", "tươi"]]),
        "location": sum([t.count(x) for x in ["gần", "ở", "tại", "khu vực"]]),
        "action": sum([t.count(x) for x in ["đặt", "liên hệ", "xem thêm", "mua"]]),
    }

    return signals


# =========================
# CTA DETECTION (GENERIC)
# =========================
def detect_cta(text: str):

    t = text.lower()

    if "đặt" in t:
        return "transaction"
    if "liên hệ" in t:
        return "contact"
    if "xem thêm" in t:
        return "explore"
    if "mua" in t or "giá" in t:
        return "commercial"

    return "informational"


# =========================
# EXTRACT CORE CONTENT (TRIM NOISE)
# =========================
def extract_core(text: str):
    """
    Không lấy full text → chỉ lấy đoạn có signal cao
    """
    sentences = text.split(".")
    important = []

    keywords = ["giá", "ngon", "tốt", "review", "top", "địa chỉ", "đặt"]

    for s in sentences:
        if any(k in s.lower() for k in keywords):
            important.append(s.strip())

    return " | ".join(important[:5])


# =========================
# MAIN NODE 2 (GENERIC + PATTERN MINING)
# =========================
async def node3_competitor_parsing(node1):

    urls = node1.get("top_urls", [])

    browser = await uc.start(headless=True)

    raw_pages = []

    try:
        for item in urls:

            url = item.get("url")
            title = item.get("title", "")

            if not url:
                continue

            try:
                page = await browser.get(url)
                await asyncio.sleep(1.5)

                soup = BeautifulSoup(await page.get_content(), "html.parser")
                text = clean_text(soup.get_text(" "))

                raw_pages.append({
                    "url": url,
                    "title": title,
                    "structure": detect_structure(text),
                    "signals": extract_signals(text),
                    "cta": detect_cta(text),
                    "core": extract_core(text),
                    "length": len(text)
                })

            except Exception as e:
                raw_pages.append({
                    "url": url,
                    "title": title,
                    "error": str(e)
                })

    finally:
        browser.stop()


    # =========================
    # AGGREGATION LAYER (QUAN TRỌNG)
    # =========================

    structure_counter = Counter([x.get("structure") for x in raw_pages if "structure" in x])

    cta_counter = Counter([x.get("cta") for x in raw_pages if "cta" in x])

    avg_length = sum([x.get("length", 0) for x in raw_pages]) / max(len(raw_pages), 1)


    return {
        "pages": raw_pages,

        # 🔥 CORE INSIGHT (KHÔNG HARD CODE)
        "market_pattern": {
            "dominant_structure": structure_counter.most_common(1)[0][0] if structure_counter else "unknown",
            "cta_distribution": dict(cta_counter),
            "avg_content_length": int(avg_length)
        }
    }


# =========================
# RUN
# =========================
node3 = await node3_competitor_parsing(node1)

print("\n===== RAW LLM OUTPUT =====\n")
node3


===== NODE 2 OUTPUT =====



{'pages': [{'url': 'https://www.bachhoaxanh.com/kinh-nghiem-hay/10-quan-hai-san-gia-re-ngon-nuc-tieng-o-da-nang-ma-ban-co-the-de-dang-tim-thay-1406465',
   'title': '10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có ...',
   'structure': 'guide',
   'signals': {'price': 51, 'quality': 76, 'location': 62, 'action': 40},
   'cta': 'transaction',
   'core': '10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy DANH MỤC SẢN PHẨM Đăng nhập #Khuyến mãi sốc #Mẹo vặt cuộc sống #Món ngon mỗi ngày #Dọn dẹp nhà xinh #Mở tiệc tại gia #Chăm sóc cá nhân Tặng 10K Mua Nước Tăng Lực Redbull Từ 250 | 2KG KT: 30/06/2026 Lấy ngay #Kinh nghiệm hay > #Mẹo vặt cuộc sống > 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng tìm thấy Nếu bạn đang tìm kiếm các quán hải sản ngon tại Đà Nẵng | Thì hãy tìm hiểu 10 quán hải sản giá rẻ ngon nức tiếng ở Đà Nẵng, mà bạn có thể dễ dàng 

In [ ]:
import re
from collections import Counter


# =========================
# CLEAN TEXT
# =========================
def normalize(text):
    return re.sub(r"\s+", " ", text.lower()).strip()


# =========================
# BUILD TEXT CORPUS
# =========================
def build_corpus(node1, node2, node15):

    texts = []

    # Node 1 (SERP titles)
    for x in node1.get("top_urls", []):
        texts.append(x.get("title", ""))

    # Node 1.5 (intent + expansion)
    texts += node15.get("expansion_signals", [])
    texts += node15.get("keyword_cluster", [])

    # Node 2 (competitors)
    for c in node2.get("competitors", []):
        texts.append(c.get("title", ""))

    return [normalize(t) for t in texts if t]


# =========================
# MARKET POSITION (INSIGHT LAYER)
# =========================
def market_position(node2):

    types = [c.get("type") for c in node2.get("competitors", [])]
    angles = []

    for c in node2.get("competitors", []):
        angles += c.get("angle", [])

    type_freq = Counter(types)
    angle_freq = Counter(angles)

    dominant_type = type_freq.most_common(1)[0][0] if type_freq else "unknown"
    dominant_angles = [k for k, _ in angle_freq.most_common(3)]

    return {
        "dominant_content_type": dominant_type,
        "dominant_angles": dominant_angles
    }


# =========================
# GAP DETECTION (WHAT IS MISSING)
# =========================
def detect_gaps(node2):

    expected = {
        "storytelling",
        "experience",
        "local_insider",
        "comparison_depth",
        "pricing_breakdown",
        "vibe_based_review"
    }

    found = set()

    for c in node2.get("competitors", []):
        found.update(c.get("angle", []))

    missing = list(expected - found)

    return missing


# =========================
# WINNING STRATEGY GENERATION
# =========================
def winning_strategy(position, gaps, node15):

    intent = node15.get("intent_scores", [])

    top_intent = intent[0][0] if intent else "unknown"

    strategy = []

    # positioning logic
    if position["dominant_content_type"] == "listicle":
        strategy.append("Avoid pure SEO listicle format")
        strategy.append("Use experience-based narrative structure")

    if "experience" in gaps:
        strategy.append("Inject real local experience storytelling")

    if "storytelling" in gaps:
        strategy.append("Add human perspective instead of generic listing")

    if "vibe_based_review" in gaps:
        strategy.append("Differentiate via atmosphere / vibe description")

    # intent alignment
    strategy.append(f"Primary intent target: {top_intent}")

    return strategy


# =========================
# MAIN NODE 3 (FINAL OUTPUT)
# =========================
def node3_gap_engine(node1, node2, node15):

    corpus = build_corpus(node1, node2, node15)

    position = market_position(node2)

    gaps = detect_gaps(node2)

    strategy = winning_strategy(position, gaps, node15)

    return {
        "market_position": position,
        "content_gaps": gaps,
        "winning_strategy": strategy
    }


# =========================
# RUN
# =========================
node3 = node3_gap_engine(node1, node2, node15)

print("\n===== RAW LLM OUTPUT =====\n")
print(node3)

In [2]:
# ============================================
# NODE 1: BASIC — Chỉ 1 endpoint, không package
# ============================================

import requests
import json
import time
import re


def get_suggestions(query: str) -> list:
    """
    Chỉ dùng 1 endpoint Google Suggest public.
    Trả về list string suggestions.
    """
    url = "https://suggestqueries.google.com/complete/search"
    
    params = {
        "client": "firefox",  # Firefox = JSON thuần, không JSONP
        "hl": "vi",
        "q": query
    }
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:109.0) Gecko/20100101 Firefox/109.0",
        "Accept": "application/json, text/javascript, */*",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
        "Referer": "https://www.google.com/"
    }
    
    try:
        response = requests.get(url, params=params, headers=headers, timeout=15)
        print(f"  Status: {response.status_code}")
        print(f"  URL: {response.url[:80]}...")
        
        if response.status_code != 200:
            print(f"  Response: {response.text[:200]}")
            return []
        
        # Firefox client trả về JSON: [query, [suggestions]]
        data = response.json()
        print(f"  Raw response type: {type(data)}")
        print(f"  Raw: {str(data)[:200]}")
        
        if isinstance(data, list) and len(data) >= 2:
            suggestions = data[1]
            if isinstance(suggestions, list):
                # Lọc bỏ query gốc và quá ngắn
                return [s for s in suggestions if s != query and len(s) > 3]
        
        return []
        
    except Exception as e:
        print(f"  ERROR: {e}")
        return []


def expand_query(query: str) -> list:
    """
    Mở rộng query để lấy nhiều suggestions.
    """
    all_results = set()
    
    # 1. Query gốc
    print(f"\n[1] Base: '{query}'")
    base = get_suggestions(query)
    all_results.update(base)
    print(f"    → {len(base)} suggestions")
    time.sleep(0.5)
    
    # 2. Thêm " " + a-z (mỗi chữ cái cho 1 suggestion mới)
    print(f"\n[2] Alphabet expansion...")
    for char in "abcdefghijklmnopqrstuvwxyz":
        new_query = f"{query} {char}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    # 3. Thêm số (giá, số người, số ngày)
    print(f"\n[3] Number expansion...")
    for num in "0123456789":
        new_query = f"{query} {num}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    # 4. Thêm từ đệm tiết lộ intent
    print(f"\n[4] Intent words...")
    intent_words = ["giá", "gần", "ở", "cho", "review", "nào", "ngon", "rẻ", "đẹp"]
    for word in intent_words:
        new_query = f"{query} {word}"
        suggs = get_suggestions(new_query)
        all_results.update(suggs)
        if suggs:
            print(f"    '{new_query[:30]}' → {len(suggs)}")
        time.sleep(0.3)
    
    return list(all_results)


def simple_intent_tag(queries: list) -> dict:
    """
    Gắn tag đơn giản — không AI.
    """
    tags = {
        "budget": ["giá", "rẻ", "tiết kiệm", "bao nhiêu", "chi phí"],
        "location": ["gần", "ở", "tại", "đường", "khu"],
        "food": ["hải sản", "món", "ăn", "nhậu", "buffet", "nướng"],
        "experience": ["review", "trải nghiệm", "ngon", "đẹp", "view"],
        "family": ["gia đình", "nhóm", "đoàn", "nhiều người"],
        "question": ["?", "là gì", "ở đâu", "có nên", "nào"]
    }
    
    result = {tag: [] for tag in tags}
    result["other"] = []
    
    for q in queries:
        q_lower = q.lower()
        matched = False
        for tag, keywords in tags.items():
            if any(kw in q_lower for kw in keywords):
                result[tag].append(q)
                matched = True
                break
        if not matched:
            result["other"].append(q)
    
    return result


# ============================================
# RUN
# ============================================

if __name__ == "__main__":
    QUERY = "nhà hàng quán nhậu hải sản Đà Nẵng"
    
    print("=" * 60)
    print("NODE 1: BASIC SUGGEST COLLECTOR")
    print("=" * 60)
    
    # Thu thập
    all_suggestions = expand_query(QUERY)
    
    print(f"\n{'='*60}")
    print(f"TOTAL: {len(all_suggestions)} unique suggestions")
    print(f"{'='*60}")
    
    # Phân loại
    tagged = simple_intent_tag(all_suggestions)
    
    print(f"\n--- TAGGED ---")
    for tag, items in sorted(tagged.items(), key=lambda x: -len(x[1])):
        if items:
            print(f"\n[{tag.upper()}] ({len(items)} items)")
            for item in items[:5]:
                print(f"  • {item}")
    
    # Lưu
    output = {
        "seed": QUERY,
        "total": len(all_suggestions),
        "suggestions": all_suggestions,
        "tagged": tagged
    }
    
    with open("node1_basic.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"\n✅ Saved: node1_basic.json")

NODE 1: BASIC SUGGEST COLLECTOR

[1] Base: 'nhà hàng quán nhậu hải sản Đà Nẵng'
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng', []]
    → 0 suggestions

[2] Alphabet expansion...
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng a', ['nhà hàng quán nhậu hải sản đà nẵng a lưới', 'nhà hàng quán nhậu hải sản đà nẵng a-z'], [], {'google:suggestdetail': [{'mp': '… ', 't': 'a lưới'}, {'mp': '… '
    'nhà hàng quán nhậu hải sản Đà ' → 2
  Status: 200
  URL: https://suggestqueries.google.com/complete/search?client=firefox&hl=vi&q=nh%C3%A...
  Raw response type: <class 'list'>
  Raw: ['nhà hàng quán nhậu hải sản Đà Nẵng b', ['nhà hàng quán nhậu hải sản đà nẵng bình dân', 'nhà hàng quán nhậu hải sản đà nẵng bao nhiêu tiền'], [

In [19]:
# ============================================
# NODE 2: COMPETITOR MINING (SERP INTELLIGENCE)
# BASIC VERSION — Chỉ parse HTML, không AI
# ============================================

import requests
import json
import time
import re
from urllib.parse import quote, urlparse
from typing import List, Dict, Any, Optional
from bs4 import BeautifulSoup
from urllib.parse import unquote, urlparse, parse_qs

# ============================================
# 1. FETCH SERP (DuckDuckGo — không anti-bot)
# ============================================

def fetch_serp(query: str, max_results: int = 10) -> List[Dict]:
    """
    Fetch SERP qua DuckDuckGo HTML — không cần API key.
    Trả về list result với title, url, snippet.
    """
    url = f"https://html.duckduckgo.com/html/?q={quote(query)}"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Accept-Language": "vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7",
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        results = []
        
        # DuckDuckGo HTML result class
        for result in soup.select(".result"):
            title_el = result.select_one(".result__a")
            snippet_el = result.select_one(".result__snippet")
            url_el = result.select_one(".result__url")
            
            if not title_el:
                continue
            
            title = title_el.get_text(strip=True)
            result_url = title_el.get("href", "")
            if "duckduckgo.com/l/" in result_url:
                parsed = urlparse(result_url)
                uddg = parse_qs(parsed.query).get("uddg", [""])
                if uddg[0]:
                    result_url = unquote(uddg[0])
            snippet = snippet_el.get_text(strip=True) if snippet_el else ""
            display_url = url_el.get_text(strip=True) if url_el else ""
            
            # Clean URL (DuckDuckGo redirect)
            if result_url.startswith("//"):
                result_url = "https:" + result_url
            
            results.append({
                "title": title,
                "url": result_url,
                "display_url": display_url,
                "snippet": snippet[:300],
                "position": len(results) + 1
            })
            
            if len(results) >= max_results:
                break
        
        return results
        
    except Exception as e:
        print(f"  ERROR fetching SERP: {e}")
        return []


# ============================================
# 2. FETCH PAGE CONTENT (Lightweight)
# ============================================

def fetch_page_content(url: str) -> Optional[Dict]:
    """
    Fetch nội dung trang — lightweight, không render JS.
    Chỉ lấy text, không lấy media.
    """
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
        
        response = requests.get(url, headers=headers, timeout=10, allow_redirects=True)
        
        if response.status_code != 200:
            return None
        
        # Check content type
        content_type = response.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return None
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Remove script/style
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        
        # Extract structure
        h1 = soup.find("h1")
        h2s = [h.get_text(strip=True) for h in soup.find_all("h2")[:5]]
        h3s = [h.get_text(strip=True) for h in soup.find_all("h3")[:5]]
        
        # First paragraph
        first_p = None
        for p in soup.find_all("p"):
            text = p.get_text(strip=True)
            if len(text) > 50:
                first_p = text[:300]
                break
        
        # Meta description
        meta_desc = ""
        meta = soup.find("meta", attrs={"name": "description"})
        if meta:
            meta_desc = meta.get("content", "")[:200]
        
        # Word count estimate
        text = soup.get_text(separator=" ", strip=True)
        word_count = len(text.split())
        
        # Detect content type
        content_type_detected = detect_content_type(soup, text)
        
        # Detect CTA
        ctas = detect_cta(soup)
        
        return {
            "url": url,
            "title": soup.title.get_text(strip=True) if soup.title else "",
            "h1": h1.get_text(strip=True) if h1 else "",
            "h2s": h2s,
            "h3s": h3s,
            "meta_description": meta_desc,
            "first_paragraph": first_p,
            "word_count": word_count,
            "content_type": content_type_detected,
            "ctas": ctas,
            "internal_links": len(soup.find_all("a", href=re.compile(r"^/|^https?://" + re.escape(urlparse(url).netloc)))),
            "external_links": len(soup.find_all("a", href=re.compile(r"^https?://"))),
            "has_images": len(soup.find_all("img")) > 0,
            "has_video": bool(soup.find("video") or soup.find("iframe", src=re.compile(r"youtube|vimeo"))),
            "has_table": bool(soup.find("table")),
            "has_list": bool(soup.find(["ul", "ol"])),
        }
        
    except Exception as e:
        print(f"    ERROR fetching {url[:50]}: {e}")
        return None


# ============================================
# 3. CONTENT TYPE DETECTION (Rule-based)
# ============================================

def detect_content_type(soup, text: str) -> str:
    """
    Detect content type từ structure — không AI.
    """
    text_lower = text.lower()
    
    # Listicle
    if any(marker in text_lower for marker in ["top ", "best ", "nhất", "danh sách", "list"]) and soup.find("ol"):
        return "listicle"
    
    # Review
    if any(marker in text_lower for marker in ["review", "đánh giá", "trải nghiệm", "cảm nhận"]) and len(text) > 2000:
        return "review"
    
    # Guide/How-to
    if any(marker in text_lower for marker in ["hướng dẫn", "cách ", "how to", "mẹo", "tip"]) and soup.find(["ol", "ul"]):
        return "guide"
    
    # Comparison
    if any(marker in text_lower for marker in ["so sánh", "vs", "hơn", "kém", "nên chọn"]) and soup.find("table"):
        return "comparison"
    
    # Directory/Listing
    if soup.find("table") and len(soup.find_all("a")) > 20:
        return "directory"
    
    # Landing page / Sales
    if len(text) < 1000 and len(soup.find_all("button", string=re.compile(r"đặt|book|mua|buy|gọi|call"))) > 0:
        return "landing_page"
    
    return "article"


# ============================================
# 4. CTA DETECTION (Rule-based)
# ============================================

def detect_cta(soup) -> List[Dict]:
    """
    Detect CTA buttons/links — pattern matching.
    """
    ctas = []
    
    # Button patterns
    button_patterns = [
        (r"đặt|book|booking|đặt phòng|đặt bàn", "booking"),
        (r"mua|buy|order|đặt hàng|gọi món", "purchase"),
        (r"gọi|call|liên hệ|contact|hotline", "contact"),
        (r"xem|view|chi tiết|tìm hiểu|read more", "information"),
        (r"tải|download|nhận|get|free", "lead_gen"),
        (r"so sánh|compare|xem giá|check price", "comparison"),
    ]
    
    for button in soup.find_all(["button", "a", "input"]):
        text = button.get_text(strip=True).lower()
        if not text:
            continue
        
        for pattern, cta_type in button_patterns:
            if re.search(pattern, text):
                ctas.append({
                    "text": text[:50],
                    "type": cta_type,
                    "element": button.name,
                    "placement": "unknown"  # Will be refined later
                })
                break
    
    # Placement detection (rough)
    body_text = soup.get_text()
    if ctas:
        first_cta_pos = body_text.lower().find(ctas[0]["text"])
        total_len = len(body_text)
        
        if first_cta_pos < total_len * 0.2:
            ctas[0]["placement"] = "top"
        elif first_cta_pos > total_len * 0.8:
            ctas[0]["placement"] = "bottom"
        else:
            ctas[0]["placement"] = "mid"
    
    return ctas[:5]  # Max 5 CTAs


# ============================================
# 5. COMPETITOR ANALYSIS (Per URL)
# ============================================

def analyze_competitor(serp_result: Dict) -> Optional[Dict]:
    """
    Phân tích 1 competitor từ SERP result.
    """
    url = serp_result["url"]
    print(f"  Analyzing: {url[:60]}...")
    
    content = fetch_page_content(url)
    if not content:
        return None
    
    # Domain analysis
    domain = urlparse(url).netloc
    
    # Detect monetization
    monetization = detect_monetization(content)
    
    # Detect tone (simple keyword matching)
    tone = detect_tone(content)
    
    return {
        "position": serp_result["position"],
        "domain": domain,
        "url": url,
        "serp_title": serp_result["title"],
        "serp_snippet": serp_result["snippet"],
        "content": content,
        "monetization": monetization,
        "tone": tone,
        "strength": infer_strength(content),
        "weakness": infer_weakness(content),
    }


def detect_monetization(content: Dict) -> List[str]:
    """
    Detect monetization model — pattern matching.
    """
    signals = []
    text = f"{content.get('title', '')} {content.get('first_paragraph', '')}".lower()
    
    if any(x in text for x in ["booking.com", "agoda", "expedia", "traveloka"]):
        signals.append("affiliate")
    
    if any(x in text for x in ["đặt ngay", "book now", "gọi điện", "hotline", "số điện thoại"]):
        signals.append("direct_booking")
    
    if any(x in text for x in ["email", "đăng ký", "nhận ưu đãi", "subscribe", "newsletter"]):
        signals.append("lead_gen")
    
    if any(x in text for x in ["quảng cáo", "sponsored", "ad", "partner"]):
        signals.append("ads")
    
    if not signals:
        signals.append("content_only")
    
    return signals


def detect_tone(content: Dict) -> Dict:
    """
    Simple tone detection — keyword frequency.
    """
    text = f"{content.get('title', '')} {' '.join(content.get('h2s', []))} {content.get('first_paragraph', '')}".lower()
    
    formal_markers = ["chúng tôi", "quý khách", "được thiết kế", "cung cấp", "dịch vụ", "chuyên nghiệp"]
    casual_markers = ["mình", "bạn", "thích", "ngon", "đẹp", "tuyệt", "chill", "đỉnh"]
    story_markers = ["hôm đó", "lần đầu", "kỷ niệm", "nhớ", "cảm giác", "trải nghiệm"]
    data_markers = ["theo thống kê", "nghiên cứu", "số liệu", "top", "xếp hạng", "điểm"]
    
    scores = {
        "formal": sum(1 for m in formal_markers if m in text),
        "casual": sum(1 for m in casual_markers if m in text),
        "storytelling": sum(1 for m in story_markers if m in text),
        "data_driven": sum(1 for m in data_markers if m in text),
    }
    
    # Normalize
    total = sum(scores.values()) or 1
    return {k: round(v/total, 2) for k, v in scores.items()}


def infer_strength(content: Dict) -> List[str]:
    """
    Infer competitor strength from content signals.
    """
    strengths = []
    
    if content.get("word_count", 0) > 2000:
        strengths.append("comprehensive_content")
    
    if content.get("has_images"):
        strengths.append("visual_content")
    
    if content.get("has_table"):
        strengths.append("structured_data")
    
    if content.get("internal_links", 0) > 5:
        strengths.append("internal_linking")
    
    if len(content.get("ctas", [])) > 2:
        strengths.append("multiple_ctas")
    
    return strengths


def infer_weakness(content: Dict) -> List[str]:
    """
    Infer competitor weakness — gap opportunity.
    """
    weaknesses = []
    
    if content.get("word_count", 0) < 500:
        weaknesses.append("thin_content")
    
    if not content.get("has_images"):
        weaknesses.append("no_visuals")
    
    if not content.get("has_video"):
        weaknesses.append("no_video")
    
    if not content.get("has_table"):
        weaknesses.append("no_comparison_table")
    
    if len(content.get("ctas", [])) == 0:
        weaknesses.append("no_cta")
    
    if content.get("content_type") == "article" and not content.get("has_list"):
        weaknesses.append("unstructured")
    
    return weaknesses


# ============================================
# 6. AGGREGATE MARKET NARRATIVE
# ============================================

def aggregate_market_narrative(competitors: List[Dict]) -> Dict:
    """
    Tổng hợp pattern từ nhiều competitor — không AI, chỉ statistics.
    """
    if not competitors:
        return {}
    
    # Content type distribution
    content_types = {}
    for c in competitors:
        ct = c["content"]["content_type"]
        content_types[ct] = content_types.get(ct, 0) + 1
    
    # Monetization distribution
    monetization = {}
    for c in competitors:
        for m in c["monetization"]:
            monetization[m] = monetization.get(m, 0) + 1
    
    # Tone average
    tone_scores = {"formal": 0, "casual": 0, "storytelling": 0, "data_driven": 0}
    for c in competitors:
        for k, v in c["tone"].items():
            tone_scores[k] += v
    tone_avg = {k: round(v/len(competitors), 2) for k, v in tone_scores.items()}
    
    # CTA patterns
    cta_types = {}
    cta_placements = {}
    for c in competitors:
        for cta in c["content"].get("ctas", []):
            cta_types[cta["type"]] = cta_types.get(cta["type"], 0) + 1
            cta_placements[cta["placement"]] = cta_placements.get(cta["placement"], 0) + 1
    
    # Common weaknesses (opportunities)
    all_weaknesses = []
    for c in competitors:
        all_weaknesses.extend(c["weakness"])
    
    weakness_opportunities = {}
    for w in all_weaknesses:
        weakness_opportunities[w] = weakness_opportunities.get(w, 0) + 1
    
    # Top opportunities (weakness xuất hiện nhiều)
    top_opportunities = [
        {"weakness": w, "count": c, "opportunity": f"Create content with {w.replace('_', ' ')}"}
        for w, c in sorted(weakness_opportunities.items(), key=lambda x: -x[1])[:5]
    ]
    
    return {
        "total_competitors_analyzed": len(competitors),
        "content_type_distribution": content_types,
        "dominant_content_type": max(content_types, key=content_types.get) if content_types else "unknown",
        "monetization_distribution": monetization,
        "dominant_monetization": max(monetization, key=monetization.get) if monetization else "unknown",
        "average_tone": tone_avg,
        "dominant_tone": max(tone_avg, key=tone_avg.get) if tone_avg else "unknown",
        "cta_patterns": {
            "types": cta_types,
            "placements": cta_placements
        },
        "average_word_count": round(sum(c["content"]["word_count"] for c in competitors) / len(competitors)),
        "content_gaps": top_opportunities,
        "serp_bias_note": f"Analyzed top {len(competitors)} results. May not represent user favorites, only SEO winners."
    }


# ============================================
# 7. MAIN ENGINE
# ============================================

def node2_engine(queries: List[str], max_results_per_query: int = 5) -> Dict:
    """
    Node 2 BASIC — SERP Intelligence.
    """
    print(f"\n{'='*60}")
    print("NODE 2: SERP INTELLIGENCE")
    print(f"{'='*60}")
    
    all_competitors = []
    all_serp_results = []
    
    for i, query in enumerate(queries, 1):
        print(f"\n[{i}/{len(queries)}] Query: '{query}'")
        
        # Fetch SERP
        serp_results = fetch_serp(query, max_results_per_query)
        print(f"  Found {len(serp_results)} SERP results")
        
        all_serp_results.extend(serp_results)
        
        # Analyze top results
        for serp in serp_results[:3]:  # Chỉ analyze top 3 để nhanh
            competitor = analyze_competitor(serp)
            if competitor:
                all_competitors.append(competitor)
                print(f"  ✓ Analyzed: {competitor['domain']}")
            time.sleep(1)  # Polite delay
        
        time.sleep(2)
    
    # Deduplicate by domain
    seen_domains = set()
    unique_competitors = []
    for c in all_competitors:
        if c["domain"] not in seen_domains:
            seen_domains.add(c["domain"])
            unique_competitors.append(c)
    
    print(f"\n{'='*60}")
    print(f"UNIQUE COMPETITORS: {len(unique_competitors)}")
    print(f"{'='*60}")
    
    # Aggregate
    market_narrative = aggregate_market_narrative(unique_competitors)
    
    # Build output
    output = {
        "input_queries": queries,
        "collection_date": time.strftime("%Y-%m-%d"),
        "stats": {
            "total_serp_results": len(all_serp_results),
            "unique_competitors": len(unique_competitors),
            "successful_fetches": len(unique_competitors),
            "failed_fetches": len(all_competitors) - len(unique_competitors)
        },
        "competitors": [
            {
                "position": c["position"],
                "domain": c["domain"],
                "url": c["url"],
                "content_type": c["content"]["content_type"],
                "word_count": c["content"]["word_count"],
                "monetization": c["monetization"],
                "tone": c["tone"],
                "strengths": c["strength"],
                "weaknesses": c["weakness"],
                "ctas": c["content"]["ctas"],
                "h2_outline": c["content"]["h2s"],
                "has_visuals": c["content"]["has_images"] or c["content"]["has_video"],
                "has_table": c["content"]["has_table"]
            }
            for c in unique_competitors
        ],
        "market_narrative": market_narrative,
        "content_opportunities": market_narrative.get("content_gaps", []),
        "next_step": "Node 3: Collect raw customer voice to validate these market claims"
    }
    
    return output


# ============================================
# 8. RUN
# ============================================

if __name__ == "__main__":
    # Input từ Node 1 — lấy top queries từ mỗi intent
    NODE1_QUERIES = [
        "nhà hàng quán nhậu hải sản Đà Nẵng giá rẻ",
        "nhà hàng quán nhậu hải sản Đà Nẵng gần biển",
        "nhà hàng quán nhậu hải sản Đà Nẵng review",
        "nhà hàng quán nhậu hải sản Đà Nẵng cho gia đình",
        "nhà hàng quán nhậu hải sản Đà Nẵng ngon nhất"
    ]
    
    result = node2_engine(NODE1_QUERIES, max_results_per_query=5)
    
    # Save
    with open("node2_output.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print("NODE 2 COMPLETE")
    print(f"Output: node2_output.json")
    print(f"{'='*60}")
    
    # Preview
    print(f"\n--- PREVIEW ---")
    print(f"Competitors: {result['stats']['unique_competitors']}")
    print(f"Dominant content type: {result['market_narrative']['dominant_content_type']}")
    print(f"Dominant monetization: {result['market_narrative']['dominant_monetization']}")
    print(f"Average tone: {result['market_narrative']['average_tone']}")
    print(f"\nTop opportunities:")
    for opp in result['content_opportunities'][:3]:
        print(f"  • {opp['weakness']} ({opp['count']} competitors) → {opp['opportunity']}")


NODE 2: SERP INTELLIGENCE

[1/5] Query: 'nhà hàng quán nhậu hải sản Đà Nẵng giá rẻ'
  Found 5 SERP results
  Analyzing: https://top10danang.com/tong-hop-top-20-quan-nhau-da-nang-ng...
  ✓ Analyzed: top10danang.com
  Analyzing: https://khamphadanang.vn/quan-hai-san-da-nang/...
    ERROR fetching https://khamphadanang.vn/quan-hai-san-da-nang/: HTTPSConnectionPool(host='khamphadanang.vn', port=443): Max retries exceeded with url: /quan-hai-san-da-nang/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)')))
  Analyzing: https://khamphadanang.vn/quan-nhau-binh-dan-da-nang/...
    ERROR fetching https://khamphadanang.vn/quan-nhau-binh-dan-da-nan: HTTPSConnectionPool(host='khamphadanang.vn', port=443): Max retries exceeded with url: /quan-nhau-binh-dan-da-nang/ (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get 

In [12]:
# ============================================
# NODE 3: SOCIAL SNAPSHOT (RAW CUSTOMER VOICE)
# BASIC VERSION — Chỉ thu thập, không summarize, không AI
# ============================================

import requests
import json
import time
import re
from urllib.parse import quote, unquote
from typing import List, Dict, Any, Optional
from bs4 import BeautifulSoup


# ============================================
# 1. FACEBOOK GROUP POSTS (Public groups only)
# ============================================

def fetch_facebook_group_posts(group_url: str, keywords: List[str], max_posts: int = 20) -> List[Dict]:
    """
    Thu thập posts từ Facebook public group.
    Chỉ lấy text public, không login.
    """
    results = []
    
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "vi-VN,vi;q=0.9",
        }
        
        response = requests.get(group_url, headers=headers, timeout=15)
        
        if response.status_code != 200:
            print(f"  Cannot access group: {response.status_code}")
            return results
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Find post containers (Facebook HTML structure changes often)
        # Fallback: extract all text blocks with keywords
        text_blocks = []
        
        for div in soup.find_all(["div", "span", "p"]):
            text = div.get_text(strip=True)
            if len(text) > 50 and len(text) < 2000:
                # Check if contains any keyword
                text_lower = text.lower()
                if any(kw.lower() in text_lower for kw in keywords):
                    text_blocks.append(text)
        
        # Deduplicate
        seen = set()
        for text in text_blocks:
            if text not in seen:
                seen.add(text)
                results.append({
                    "id": f"fb_{len(results):03d}",
                    "text": text,
                    "type": classify_raw_type(text),
                    "severity": classify_severity(text),
                    "topic_hint": extract_topic_hints(text, keywords),
                    "audience": classify_audience(text),
                    "engagement": 0,  # Cannot get without API
                    "timestamp": "unknown",
                    "platform": "facebook_group",
                    "url": group_url,
                    "source_name": extract_group_name(group_url)
                })
                
                if len(results) >= max_posts:
                    break
        
        return results
        
    except Exception as e:
        print(f"  ERROR fetching Facebook group: {e}")
        return results


def extract_group_name(url: str) -> str:
    """Extract group name from URL"""
    match = re.search(r'groups/([^/]+)', url)
    return match.group(1) if match else "unknown"


# ============================================
# 2. REDDIT THREADS (Public, no API key needed)
# ============================================

def fetch_reddit_threads(subreddit: str, keywords: List[str], max_posts: int = 20) -> List[Dict]:
    """
    Thu thập threads từ Reddit public JSON API.
    Không cần API key.
    """
    results = []
    
    # Build search query
    search_query = " OR ".join(keywords[:3])  # Max 3 keywords for Reddit search
    
    urls = [
        f"https://www.reddit.com/r/{subreddit}/search.json?q={quote(search_query)}&restrict_sr=1&sort=new&limit=25",
        f"https://www.reddit.com/r/{subreddit}/new.json?limit=25"  # Fallback: latest posts
    ]
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept": "application/json",
    }
    
    for url in urls:
        try:
            response = requests.get(url, headers=headers, timeout=15)
            
            if response.status_code != 200:
                continue
            
            data = response.json()
            posts = data.get("data", {}).get("children", [])
            
            for post in posts:
                post_data = post.get("data", {})
                
                title = post_data.get("title", "")
                selftext = post_data.get("selftext", "")
                full_text = f"{title}. {selftext}".strip()
                
                if not full_text or len(full_text) < 50:
                    continue
                
                # Check keywords
                text_lower = full_text.lower()
                if not any(kw.lower() in text_lower for kw in keywords):
                    continue
                
                results.append({
                    "id": f"reddit_{post_data.get('id', len(results)):s}",
                    "text": full_text[:1000],  # Truncate very long posts
                    "type": classify_raw_type(full_text),
                    "severity": classify_severity(full_text),
                    "topic_hint": extract_topic_hints(full_text, keywords),
                    "audience": classify_audience(full_text),
                    "engagement": post_data.get("score", 0) + post_data.get("num_comments", 0),
                    "timestamp": time.strftime("%Y-%m-%d", time.localtime(post_data.get("created_utc", 0))) if post_data.get("created_utc") else "unknown",
                    "platform": "reddit",
                    "url": f"https://reddit.com{post_data.get('permalink', '')}",
                    "source_name": f"r/{subreddit}"
                })
                
                if len(results) >= max_posts:
                    break
            
            if results:
                break  # Got results, no need fallback
                
        except Exception as e:
            print(f"  ERROR fetching Reddit: {e}")
            continue
    
    return results


# ============================================
# 3. GOOGLE REVIEWS (Via search snippet)
# ============================================

def fetch_google_reviews(business_name: str, location: str, max_reviews: int = 10) -> List[Dict]:
    """
    Thu thập reviews từ Google search snippets.
    Not perfect but works for BASIC.
    """
    results = []
    
    query = f"{business_name} {location} review"
    url = f"https://www.google.com/search?q={quote(query)}&hl=vi"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Look for review-like text blocks
        review_blocks = []
        
        # Pattern 1: Review snippets in search results
        for div in soup.find_all("div"):
            text = div.get_text(strip=True)
            # Review patterns: stars, "review", rating indicators
            if any(marker in text for marker in ["★", "☆", "review", "đánh giá", "tuyệt vời", "tệ", "kém"]):
                if 50 < len(text) < 500:
                    review_blocks.append(text)
        
        # Pattern 2: Common review structures
        for span in soup.find_all(["span", "div"]):
            text = span.get_text(strip=True)
            if re.search(r'\d+\s*đánh giá|\d+\s*review', text):
                continue  # Skip count headers
            if len(text) > 80 and any(m in text for m in ["thích", "không thích", "nên", "không nên", "tốt", "xấu"]):
                review_blocks.append(text)
        
        # Deduplicate and classify
        seen = set()
        for text in review_blocks:
            if text not in seen and len(results) < max_reviews:
                seen.add(text)
                results.append({
                    "id": f"google_review_{len(results):03d}",
                    "text": text,
                    "type": classify_raw_type(text),
                    "severity": classify_severity(text),
                    "topic_hint": extract_topic_hints(text, [business_name, location]),
                    "audience": "unknown",
                    "engagement": 0,
                    "timestamp": "unknown",
                    "platform": "google_reviews",
                    "url": url,
                    "source_name": f"Google: {business_name}"
                })
        
        return results
        
    except Exception as e:
        print(f"  ERROR fetching Google reviews: {e}")
        return results


# ============================================
# 4. CLASSIFICATION — Rule-based, NO LLM
# ============================================

def classify_raw_type(text: str) -> str:
    """
    Phân loại type dựa trên keyword — không AI.
    """
    text_lower = text.lower()
    
    # Complaint indicators
    complaint_markers = ["nhưng", "tuy nhiên", "không", "thiếu", "tệ", "lừa", "buồn", "thất vọng", 
                        "kém", "dở", "mắc", "đắt", "không đáng", "tránh", "đừng", "không nên",
                        "tệ hại", "kinh khủng", "horrible", "bad", "worst", "disappointed"]
    
    # Praise indicators
    praise_markers = ["tuyệt", "đỉnh", "thích", "nên", "recommend", "❤️", "👍", "😍", "🥰",
                   "ngon", "đẹp", "sạch", "chu đáo", "nhiệt tình", "tuyệt vời", "perfect",
                   "best", "amazing", "great", "love", "favorite"]
    
    # Question indicators
    question_markers = ["?", "cho hỏi", "ai biết", "làm sao", "có nên", "nên không", "có không",
                       "ở đâu", "bao nhiêu", "là gì", "như thế nào", "how to", "what", "where"]
    
    # Comparison indicators
    comparison_markers = ["vs", "so với", "hơn", "kém", "nên chọn", "compare", "so sánh",
                         "hay hơn", "tốt hơn", "rẻ hơn", "đắt hơn"]
    
    # Story/experience indicators
    story_markers = ["mình đã", "hôm đó", "lần đầu", "kỷ niệm", "nhớ", "cảm giác", "trải nghiệm",
                    "đi", "ăn", "ở", "check-in", "đặt", "gọi món", "vào", "ra"]
    
    # Tip/advice indicators
    tip_markers = ["nên", "mẹo", "nhớ", "đừng", "chú ý", "lưu ý", "tip", "advice", "khuyên",
                  "nếu", "thì", "hãy", "nên đi", "nên thử"]
    
    scores = {
        "complaint": sum(1 for m in complaint_markers if m in text_lower),
        "praise": sum(1 for m in praise_markers if m in text_lower),
        "question": sum(1 for m in question_markers if m in text_lower),
        "comparison": sum(1 for m in comparison_markers if m in text_lower),
        "story": sum(1 for m in story_markers if m in text_lower),
        "tip": sum(1 for m in tip_markers if m in text_lower),
    }
    
    # Return highest score, default to "story"
    if max(scores.values()) == 0:
        return "story"
    
    return max(scores, key=scores.get)


def classify_severity(text: str) -> str:
    """
    Phân loại severity — rule-based.
    """
    text_lower = text.lower()
    
    # Deal breaker
    deal_breakers = ["không bao giờ", "lừa đảo", "tệ hơn cả", "đòi refund", "kiện", "cảnh sát",
                  "never again", "scam", "worst ever", "disaster", "terrible"]
    
    # Major
    major = ["rất thất vọng", "không như mong đợi", "giá quá cao", "không đáng", "kém",
            "very disappointed", "not worth", "overpriced", "bad experience"]
    
    # Enthusiastic
    enthusiastic = ["đỉnh của chóp", "best ever", "phải đi", "không thể bỏ lỡ", "nhất định",
                   "must try", "life changing", "heaven", "perfect", "amazing"]
    
    if any(m in text_lower for m in deal_breakers):
        return "deal_breaker"
    elif any(m in text_lower for m in major):
        return "major"
    elif any(m in text_lower for m in enthusiastic):
        return "enthusiastic"
    elif "?" in text:
        return "informational"
    else:
        return "minor"  # Default for praise/tip without strong emotion


def extract_topic_hints(text: str, keywords: List[str]) -> List[str]:
    """
    Extract topic hints từ text — keyword matching.
    """
    text_lower = text.lower()
    hints = []
    
    topic_keywords = {
        "price": ["giá", "tiền", "đắt", "rẻ", "mắc", "bình dân", "hợp lý", "budget", "cost", "price"],
        "location": ["gần", "xa", "đường", "khu", "vị trí", "trung tâm", "biển", "location", "address"],
        "food_quality": ["ngon", "dở", "tươi", "khô", "mặn", "nhạt", "vị", "taste", "flavor", "fresh"],
        "service": ["phục vụ", "nhân viên", "chủ quán", "thái độ", "nhiệt tình", "lạnh nhạt", "service", "staff"],
        "cleanliness": ["sạch", "bẩn", "chuẩn", "gọn", "clean", "dirty", "hygiene"],
        "atmosphere": ["không gian", "view", "decor", "yên tĩnh", "ồn", "ambiance", "music", "noise"],
        "wait_time": ["chờ", "lâu", "nhanh", "slow", "fast", "wait", "quick"],
        "portion": ["nhiều", "ít", "no", "đầy", "đủ", "portion", "size", "full"],
        "family": ["gia đình", "trẻ em", "bé", "con", "nhóm", "đoàn", "family", "kids", "children"],
        "parking": ["đỗ xe", "gửi xe", "parking", "lot"],
    }
    
    for topic, topic_kws in topic_keywords.items():
        if any(kw in text_lower for kw in topic_kws):
            hints.append(topic)
    
    # Also add matching keywords from input
    for kw in keywords:
        if kw.lower() in text_lower and kw.lower() not in hints:
            hints.append(kw.lower())
    
    return hints[:5]  # Max 5 hints


def classify_audience(text: str) -> str:
    """
    Classify audience — rule-based.
    """
    text_lower = text.lower()
    
    if any(m in text_lower for m in ["bé", "con", "trẻ em", "gia đình", "nhóm", "đoàn", "family", "kids"]):
        return "family_with_kids"
    elif any(m in text_lower for m in ["vợ chồng", "honeymoon", "2 người", "couple", "date", "romantic"]):
        return "young_couple"
    elif any(m in text_lower for m in ["mình đi một mình", "backpacker", "alone", "solo", "một mình"]):
        return "solo_traveler"
    elif any(m in text_lower for m in ["công tác", "meeting", "business", "work", "conference"]):
        return "business"
    elif any(m in text_lower for m in ["bạn bè", "nhóm bạn", "tụ tập", "nhậu", "nhậu nhẹt", "party", "friends"]):
        return "group_friends"
    elif any(m in text_lower for m in ["lớn tuổi", "ba mẹ", "ông bà", "senior", "elderly", "old"]):
        return "senior"
    else:
        return "general"


# ============================================
# 5. AGGREGATE — Statistics only, NO LLM
# ============================================

def aggregate_social_truth(raw_voices: List[Dict]) -> Dict:
    """
    Tổng hợp statistics từ raw voices — KHÔNG summarize, KHÔNG AI.
    """
    if not raw_voices:
        return {}
    
    # Count by type
    type_counts = {}
    for v in raw_voices:
        t = v["type"]
        type_counts[t] = type_counts.get(t, 0) + 1
    
    # Count by severity
    severity_counts = {}
    for v in raw_voices:
        s = v["severity"]
        severity_counts[s] = severity_counts.get(s, 0) + 1
    
    # Extract recurring words (simple frequency)
    all_text = " ".join([v["text"].lower() for v in raw_voices])
    words = re.findall(r'\b[a-zA-ZÀ-ỹ]+\b', all_text)
    
    # Filter common stop words
    stop_words = {"là", "có", "và", "của", "để", "một", "những", "các", "được", "trong", "cho", "với", "không", "rất", "này", "đó", "thì", "lại", "cũng", "nên", "đã", "sẽ", "đang", "còn", "nhưng", "tuy", "vì", "nếu", "the", "and", "for", "are", "with", "was", "you", "your", "this", "that", "have", "from", "they", "know", "want", "been", "good", "much", "some", "time"}
    
    word_freq = {}
    for w in words:
        if len(w) > 2 and w not in stop_words:
            word_freq[w] = word_freq.get(w, 0) + 1
    
    # Pain words (negative context)
    pain_words = ["tệ", "dở", "đắt", "mắc", "chờ", "lâu", "bẩn", "ồn", "lạnh nhạt", "không nhiệt tình", "không tươi", "khô", "nhạt", "mặn", "khó", "phiền", "thất vọng", "kém", "dơ", "chật", "nóng", "lạnh"]
    recurring_pain = [w for w in pain_words if word_freq.get(w, 0) >= 2]
    
    # Praise words (positive context)
    praise_words = ["ngon", "tươi", "sạch", "nhiệt tình", "chu đáo", "rẻ", "hợp lý", "nhanh", "đẹp", "yên tĩnh", "thoáng", "rộng", "view", "đẹp", "thích", "tuyệt", "đỉnh"]
    recurring_praise = [w for w in praise_words if word_freq.get(w, 0) >= 2]
    
    # Unmet needs (question + complaint patterns)
    unmet = []
    for v in raw_voices:
        if v["type"] in ["question", "complaint"]:
            # Extract need from text
            text = v["text"].lower()
            if "có" in text and "không" in text:
                # "Có ... không" = unmet need
                match = re.search(r'có\s+(.+?)\s+không', text)
                if match:
                    unmet.append(match.group(1).strip())
    
    # Source diversity
    sources = {}
    for v in raw_voices:
        src = v.get("source_name", "unknown")
        sources[src] = sources.get(src, 0) + 1
    
    # Confidence assessment
    confidence = "high" if len(sources) >= 3 and len(raw_voices) >= 20 else \
                 "medium" if len(sources) >= 2 and len(raw_voices) >= 10 else "low"
    
    return {
        "total_voices": len(raw_voices),
        "source_diversity": len(sources),
        "sources_breakdown": sources,
        "type_distribution": type_counts,
        "severity_distribution": severity_counts,
        "recurring_pain_words": recurring_pain,
        "recurring_praise_words": recurring_praise,
        "unmet_needs": list(dict.fromkeys(unmet))[:10],  # Deduplicate, max 10
        "confidence": confidence,
        "bias_flags": [
            "pre_purchase_focus" if type_counts.get("question", 0) > type_counts.get("story", 0) else None,
            "family_heavy" if sum(1 for v in raw_voices if v["audience"] == "family_with_kids") > len(raw_voices) * 0.5 else None,
            "complaint_skew" if type_counts.get("complaint", 0) > len(raw_voices) * 0.6 else None,
            "praise_skew" if type_counts.get("praise", 0) > len(raw_voices) * 0.6 else None,
        ],
        "collection_date": time.strftime("%Y-%m-%d")
    }


# ============================================
# 6. MAIN ENGINE
# ============================================

def node3_engine(keywords: List[str], sources_config: Dict) -> Dict:
    """
    Node 3 BASIC — Social Snapshot.
    """
    print(f"\n{'='*60}")
    print("NODE 3: SOCIAL SNAPSHOT (RAW CUSTOMER VOICE)")
    print(f"{'='*60}")
    
    all_voices = []
    
    # 1. Facebook Groups
    fb_groups = sources_config.get("facebook_groups", [])
    for i, group_url in enumerate(fb_groups, 1):
        print(f"\n[{i}/{len(fb_groups)}] Facebook Group: {group_url[:50]}...")
        voices = fetch_facebook_group_posts(group_url, keywords, max_posts=15)
        all_voices.extend(voices)
        print(f"  Collected: {len(voices)} raw voices")
        time.sleep(2)
    
    # 2. Reddit
    subreddits = sources_config.get("subreddits", [])
    for i, sub in enumerate(subreddits, 1):
        print(f"\n[{i}/{len(subreddits)}] Reddit: r/{sub}...")
        voices = fetch_reddit_threads(sub, keywords, max_posts=15)
        all_voices.extend(voices)
        print(f"  Collected: {len(voices)} raw voices")
        time.sleep(2)
    
    # 3. Google Reviews (for specific businesses)
    businesses = sources_config.get("businesses", [])
    for i, biz in enumerate(businesses, 1):
        print(f"\n[{i}/{len(businesses)}] Google Reviews: {biz}...")
        voices = fetch_google_reviews(biz, "Đà Nẵng", max_reviews=8)
        all_voices.extend(voices)
        print(f"  Collected: {len(voices)} raw voices")
        time.sleep(1)
    
    # Aggregate
    print(f"\n{'='*60}")
    print(f"TOTAL RAW VOICES: {len(all_voices)}")
    print(f"{'='*60}")
    
    truth_assessment = aggregate_social_truth(all_voices)
    
    # Build output
    output = {
        "keywords": keywords,
        "collection_date": time.strftime("%Y-%m-%d"),
        "stats": {
            "total_raw_voices": len(all_voices),
            "sources": truth_assessment.get("source_diversity", 0),
            "complaints": truth_assessment.get("type_distribution", {}).get("complaint", 0),
            "praises": truth_assessment.get("type_distribution", {}).get("praise", 0),
            "questions": truth_assessment.get("type_distribution", {}).get("question", 0),
        },
        "raw_voices": all_voices,
        "source_truth_assessment": truth_assessment,
        "next_step": "Node 4: Cross-validate with Node 2 market narrative"
    }
    
    return output


# ============================================
# 7. RUN
# ============================================

if __name__ == "__main__":
    # Keywords từ Node 1
    KEYWORDS = [
        "nhà hàng quán nhậu hải sản Đà Nẵng",
        "hải sản Đà Nẵng",
        "quán nhậu Đà Nẵng",
        "ăn hải sản Đà Nẵng giá rẻ",
        "nhà hàng gia đình Đà Nẵng"
    ]
    
    # Sources config — tìm public groups/pages
    SOURCES = {
        "facebook_groups": [
            # Thay bằng URL thực tế của public groups
            # "https://facebook.com/groups/du-lich-da-nang",
            # "https://facebook.com/groups/review-da-nang",
        ],
        "subreddits": [
            "VietNam",
            "travel",
            "solotravel",
            "food",
        ],
        "businesses": [
            # Thay bằng tên nhà hàng thực tế
            # "Muoi Bien Seafood",
            # "Hai San Be Man",
        ]
    }
    
    # Note: Facebook groups cần public URLs. Nếu không có, dùng Reddit + Google Reviews.
    # Hoặc thu thập thủ công từ Facebook rồi lưu vào file JSON.
    
    result = node3_engine(KEYWORDS, SOURCES)
    
    # Save
    with open("node3_output.json", "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print("NODE 3 COMPLETE")
    print(f"Output: node3_output.json")
    print(f"{'='*60}")
    
    # Preview
    print(f"\n--- PREVIEW ---")
    print(f"Total voices: {result['stats']['total_raw_voices']}")
    print(f"Sources: {result['stats']['sources']}")
    print(f"Complaints: {result['stats']['complaints']}")
    print(f"Praises: {result['stats']['praises']}")
    print(f"Questions: {result['stats']['questions']}")
    
    if result['raw_voices']:
        print(f"\nSample voices:")
        for v in result['raw_voices'][:3]:
            print(f"\n  [{v['type'].upper()} | {v['severity']}]")
            print(f"  Platform: {v['platform']}")
            print(f"  Text: {v['text'][:150]}...")
            print(f"  Topics: {v['topic_hint']}")


NODE 3: SOCIAL SNAPSHOT (RAW CUSTOMER VOICE)

[1/4] Reddit: r/VietNam...
  Collected: 0 raw voices

[2/4] Reddit: r/travel...
  Collected: 0 raw voices

[3/4] Reddit: r/solotravel...
  Collected: 0 raw voices

[4/4] Reddit: r/food...
  Collected: 0 raw voices

TOTAL RAW VOICES: 0

NODE 3 COMPLETE
Output: node3_output.json

--- PREVIEW ---
Total voices: 0
Sources: 0
Complaints: 0
Praises: 0
Questions: 0


In [15]:
# ============================================
# NODE 3: SOCIAL SNAPSHOT — ĐỌC MANUAL INPUT
# ============================================

import json
import time
from typing import List, Dict, Any


def load_manual_input(filename: str = "node3_manual_input.json") -> List[Dict]:
    """
    Đọc manual input từ file.
    """
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
            return data.get("raw_voices", [])
    except FileNotFoundError:
        print(f"  ⚠️ {filename} not found. Create it with manual data.")
        return []
    except json.JSONDecodeError:
        print(f"  ⚠️ {filename} invalid JSON.")
        return []


def aggregate_social_truth(raw_voices: List[Dict]) -> Dict:
    """
    Tổng hợp statistics — KHÔNG LLM.
    """
    if not raw_voices:
        return {
            "total_voices": 0,
            "source_diversity": 0,
            "confidence": "low",
            "recurring_pain_words": [],
            "recurring_praise_words": [],
            "unmet_needs": []
        }
    
    # Count by type
    type_counts = {}
    for v in raw_voices:
        t = v.get("type", "story")
        type_counts[t] = type_counts.get(t, 0) + 1
    
    # Count by severity
    severity_counts = {}
    for v in raw_voices:
        s = v.get("severity", "minor")
        severity_counts[s] = severity_counts.get(s, 0) + 1
    
    # Extract recurring words
    all_text = " ".join([v.get("text", "").lower() for v in raw_voices])
    
    # Pain words
    pain_words = ["tệ", "dở", "đắt", "mắc", "chờ", "lâu", "bẩn", "ồn", "lạnh nhạt", 
                 "không nhiệt tình", "không tươi", "khô", "nhạt", "mặn", "khó", "phiền", 
                 "thất vọng", "kém", "dơ", "chật", "nóng", "lạnh", "lừa", "scam", "fake"]
    recurring_pain = [w for w in pain_words if w in all_text]
    
    # Praise words
    praise_words = ["ngon", "tươi", "sạch", "nhiệt tình", "chu đáo", "rẻ", "hợp lý", 
                   "nhanh", "đẹp", "yên tĩnh", "thoáng", "rộng", "view", "thích", 
                   "tuyệt", "đỉnh", "amazing", "perfect", "love", "great"]
    recurring_praise = [w for w in praise_words if w in all_text]
    
    # Unmet needs từ question + complaint
    unmet = []
    for v in raw_voices:
        if v.get("type") in ["question", "complaint"]:
            text = v.get("text", "").lower()
            # Pattern "có ... không"
            import re
            matches = re.findall(r'có\s+(.+?)\s+không', text)
            unmet.extend(matches)
            # Pattern "hỏi ... nào"
            matches2 = re.findall(r'hỏi\s+(.+?)\s+nào', text)
            unmet.extend(matches2)
    
    # Source diversity
    sources = {}
    for v in raw_voices:
        src = v.get("source_name", "unknown")
        sources[src] = sources.get(src, 0) + 1
    
    # Confidence
    confidence = "high" if len(sources) >= 3 and len(raw_voices) >= 10 else \
                 "medium" if len(sources) >= 2 and len(raw_voices) >= 5 else "low"
    
    # Bias flags
    bias_flags = []
    if type_counts.get("question", 0) > type_counts.get("story", 0):
        bias_flags.append("pre_purchase_focus")
    if sum(1 for v in raw_voices if v.get("audience") == "family_with_kids") > len(raw_voices) * 0.5:
        bias_flags.append("family_heavy")
    if type_counts.get("complaint", 0) > len(raw_voices) * 0.6:
        bias_flags.append("complaint_skew")
    if type_counts.get("praise", 0) > len(raw_voices) * 0.6:
        bias_flags.append("praise_skew")
    
    return {
        "total_voices": len(raw_voices),
        "source_diversity": len(sources),
        "sources_breakdown": sources,
        "type_distribution": type_counts,
        "severity_distribution": severity_counts,
        "recurring_pain_words": list(dict.fromkeys(recurring_pain)),
        "recurring_praise_words": list(dict.fromkeys(recurring_praise)),
        "unmet_needs": list(dict.fromkeys(unmet))[:10],
        "confidence": confidence,
        "bias_flags": bias_flags,
        "collection_date": time.strftime("%Y-%m-%d")
    }


def node3_manual_engine(filename: str = "node3_manual_input.json") -> Dict:
    """
    Node 3 — Đọc manual input, output JSON.
    """
    print(f"\n{'='*60}")
    print("NODE 3: SOCIAL SNAPSHOT (MANUAL INPUT)")
    print(f"{'='*60}")
    
    # Load manual input
    print(f"\n→ Loading manual input from {filename}...")
    raw_voices = load_manual_input(filename)
    print(f"  Loaded: {len(raw_voices)} raw voices")
    
    # Aggregate
    print("\n→ Aggregating social truth...")
    truth = aggregate_social_truth(raw_voices)
    
    # Build output
    output = {
        "input_source": filename,
        "collection_date": time.strftime("%Y-%m-%d"),
        "stats": {
            "total_raw_voices": len(raw_voices),
            "sources": truth.get("source_diversity", 0),
            "complaints": truth.get("type_distribution", {}).get("complaint", 0),
            "praises": truth.get("type_distribution", {}).get("praise", 0),
            "questions": truth.get("type_distribution", {}).get("question", 0),
            "tips": truth.get("type_distribution", {}).get("tip", 0),
            "confidence": truth.get("confidence", "low")
        },
        "raw_voices": raw_voices,
        "source_truth_assessment": truth,
        "next_step": "Node 4: Cross-validate with Node 2 market narrative"
    }
    
    # Save
    with open("node3_output.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print("NODE 3 COMPLETE")
    print(f"Output: node3_output.json")
    print(f"{'='*60}")
    
    # Preview
    print(f"\n--- PREVIEW ---")
    print(f"Total voices: {output['stats']['total_raw_voices']}")
    print(f"Sources: {output['stats']['sources']}")
    print(f"Complaints: {output['stats']['complaints']}")
    print(f"Praises: {output['stats']['praises']}")
    print(f"Questions: {output['stats']['questions']}")
    print(f"Tips: {output['stats']['tips']}")
    print(f"Confidence: {output['stats']['confidence']}")
    
    if truth.get("recurring_pain_words"):
        print(f"\nPain words: {truth['recurring_pain_words'][:5]}")
    if truth.get("recurring_praise_words"):
        print(f"Praise words: {truth['recurring_praise_words'][:5]}")
    if truth.get("unmet_needs"):
        print(f"Unmet needs: {truth['unmet_needs'][:3]}")
    
    return output


if __name__ == "__main__":
    node3_manual_engine()


NODE 3: SOCIAL SNAPSHOT (MANUAL INPUT)

→ Loading manual input from node3_manual_input.json...
  Loaded: 7 raw voices

→ Aggregating social truth...

NODE 3 COMPLETE
Output: node3_output.json

--- PREVIEW ---
Total voices: 7
Sources: 7
Complaints: 3
Praises: 2
Questions: 1
Tips: 1
Confidence: medium

Pain words: ['đắt', 'khô', 'khó', 'thất vọng', 'lạnh']
Praise words: ['ngon', 'tươi', 'rẻ', 'hợp lý', 'nhanh']
Unmet needs: ['đắt', 'quán hải sản']


In [18]:
# ============================================
# NODE 4 FIX: CROSS-VALIDATION + MANUAL CLAIMS
# ============================================

import json
import time
import re
from typing import List, Dict, Any, Optional


# ============================================
# 1. LOAD INPUT
# ============================================

def load_node_output(filename: str) -> Dict:
    try:
        with open(filename, "r", encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"  ⚠️ {filename} not found.")
        return {}
    except json.JSONDecodeError:
        print(f"  ⚠️ {filename} invalid JSON.")
        return {}


# ============================================
# 2. EXTRACT CLAIMS — HARDENED
# ============================================

def extract_market_claims(node2: Dict) -> List[Dict]:
    """
    Extract claims từ Node 2 — bao gồm cả hardcoded claims nếu Node 2 rỗng.
    """
    claims = []
    
    # Từ competitor data
    for comp in node2.get("competitors", []):
        # Title claim
        title = comp.get("serp_title", "")
        if title:
            claims.append({
                "claim": title,
                "source": comp.get("domain", "unknown"),
                "type": "title_promise"
            })
        
        # Content type claim
        ct = comp.get("content_type", "")
        if ct:
            claims.append({
                "claim": f"Market uses {ct} format for seafood content",
                "source": comp.get("domain", "unknown"),
                "type": "content_format"
            })
        
        # Monetization claim
        for m in comp.get("monetization", []):
            claims.append({
                "claim": f"Competitor uses {m} monetization",
                "source": comp.get("domain", "unknown"),
                "type": "business_model"
            })
    
    # Từ market narrative
    narrative = node2.get("market_narrative", {})
    if narrative.get("dominant_content_type"):
        claims.append({
            "claim": f"Dominant content: {narrative['dominant_content_type']}",
            "source": "market_narrative",
            "type": "trend"
        })
    
    # HARDCODED CLAIMS cho test — dựa trên thực tế market
    hardcoded_claims = [
        {"claim": "Hải sản Đà Nẵng giá rẻ, tươi sống", "source": "market_general", "type": "price_quality"},
        {"claim": "View biển đẹp, không gian sang trọng", "source": "market_general", "type": "experience"},
        {"claim": "Phục vụ nhiệt tình, chu đáo", "source": "market_general", "type": "service"},
        {"claim": "Tôm hùm tươi sống 800k/kg", "source": "market_general", "type": "specific_offer"},
        {"claim": "Không phí ẩn, giá niêm yết", "source": "market_general", "type": "pricing"},
        {"claim": "Gần biển Mỹ Khê, đi bộ 5 phút", "source": "market_general", "type": "location"},
    ]
    
    # Chỉ thêm hardcoded nếu claims từ Node 2 ít
    if len(claims) < 3:
        claims.extend(hardcoded_claims)
        print(f"  Added {len(hardcoded_claims)} hardcoded claims for testing")
    
    return claims


def extract_customer_truths(node3: Dict) -> List[Dict]:
    """
    Extract truths từ Node 3 raw voices.
    """
    truths = []
    
    for voice in node3.get("raw_voices", []):
        truths.append({
            "id": voice.get("id", ""),
            "text": voice.get("text", ""),
            "type": voice.get("type", ""),
            "severity": voice.get("severity", ""),
            "topics": voice.get("topic_hint", []),
            "audience": voice.get("audience", "")
        })
    
    return truths


# ============================================
# 3. CROSS-VALIDATION — MỞ RỘNG
# ============================================

def is_contradiction(claim: str, truth: str) -> bool:
    """
    Detect contradiction — mở rộng pattern matching.
    """
    claim_lower = claim.lower()
    truth_lower = truth.lower()
    
    # === PRICE CONTRADICTIONS ===
    # Claim "giá rẻ" vs truth "đắt", "mắc", "phụ thu", "hidden fee"
    price_positive = ["giá rẻ", "rẻ", "hợp lý", "phải chăng", "affordable", "cheap", "budget", "tiết kiệm"]
    price_negative = ["đắt", "mắc", "phụ thu", "phí ẩn", "hidden fee", "service charge", "vat", "không đáng", "bill cao", "tổng bill", "1.2m", "1.2 triệu", "2.8 triệu", "gấp đôi", "expensive", "overpriced", "scam"]
    
    if any(p in claim_lower for p in price_positive):
        if any(n in truth_lower for n in price_negative):
            return True
    
    # Claim "không phí ẩn" vs truth "phí ẩn"
    if any(p in claim_lower for p in ["không phí", "giá niêm yết", "no hidden", "transparent"]):
        if any(n in truth_lower for n in ["phí ẩn", "phụ thu", "service charge", "hidden"]):
            return True
    
    # === FRESHNESS CONTRADICTIONS ===
    # Claim "tươi sống" vs truth "đông lạnh", "không tươi"
    fresh_positive = ["tươi sống", "tươi", "fresh", "sống", "vừa bắt", "mới đánh"]
    fresh_negative = ["đông lạnh", "frozen", "không tươi", "khô", "cứng", "nhũn", "ươn", "hôi", "stale", "not fresh"]
    
    if any(p in claim_lower for p in fresh_positive):
        if any(n in truth_lower for n in fresh_negative):
            return True
    
    # === VIEW/LOCATION CONTRADICTIONS ===
    # Claim "view biển" vs truth "view tường"
    view_positive = ["view biển", "view đẹp", "nhìn biển", "gần biển", "sát biển", "ocean view", "sea view", "beachfront"]
    view_negative = ["view tường", "không thấy biển", "xa biển", "nhìn tường", "no view", "wall", "không gần"]
    
    if any(p in claim_lower for p in view_positive):
        if any(n in truth_lower for n in view_negative):
            return True
    
    # === SERVICE CONTRADICTIONS ===
    # Claim "phục vụ tốt" vs truth "lạnh nhạt"
    service_positive = ["phục vụ tốt", "nhiệt tình", "chu đáo", "good service", "friendly", "attentive", "nhớ tên", "care"]
    service_negative = ["lạnh nhạt", "thái độ", "rude", "ignore", "bad service", "không quan tâm", "chậm", "không nhiệt tình"]
    
    if any(p in claim_lower for p in service_positive):
        if any(n in truth_lower for n in service_negative):
            return True
    
    # === CLEANLINESS CONTRADICTIONS ===
    clean_positive = ["sạch", "clean", "chuẩn", "hygiene"]
    clean_negative = ["bẩn", "dirty", "dơ", "messy", "chuột", "côn trùng"]
    
    if any(p in claim_lower for p in clean_positive):
        if any(n in truth_lower for n in clean_negative):
            return True
    
    # === FOOD QUALITY CONTRADICTIONS ===
    food_positive = ["ngon", "delicious", "tuyệt", "perfect", "amazing", "best"]
    food_negative = ["dở", "nhạt", "mặn", "khô", "không ngon", "bad taste", "disappointed"]
    
    if any(p in claim_lower for p in food_positive):
        if any(n in truth_lower for n in food_negative):
            return True
    
    return False


def is_validation(claim: str, truth: str) -> bool:
    """
    Detect validation — cả claim và truth cùng positive.
    """
    claim_lower = claim.lower()
    truth_lower = truth.lower()
    
    positive_words = ["ngon", "tươi", "sạch", "nhiệt tình", "đẹp", "good", "great", "amazing", "love", "perfect", "tuyệt", "đỉnh", "chu đáo", "thích"]
    
    claim_pos = any(w in claim_lower for w in positive_words)
    truth_pos = any(w in truth_lower for w in positive_words)
    
    if claim_pos and truth_pos:
        # Check same topic
        topics = ["hải sản", "phục vụ", "view", "giá", "không gian", "vị trí", "tôm", "cua", "mực"]
        for topic in topics:
            if topic in claim_lower and topic in truth_lower:
                return True
    
    return False


def infer_opportunity(claim: Dict, truth: Dict) -> str:
    """
    Infer opportunity từ mismatch.
    """
    claim_text = claim.get("claim", "").lower()
    truth_text = truth.get("text", "").lower()
    
    if any(w in claim_text for w in ["giá rẻ", "rẻ", "affordable"]) and any(w in truth_text for w in ["đắt", "mắc", "phụ thu", "phí ẩn"]):
        return "transparent_pricing: show total cost including all fees"
    
    if any(w in claim_text for w in ["tươi sống", "fresh", "tươi"]) and any(w in truth_text for w in ["đông lạnh", "frozen", "không tươi", "khô"]):
        return "freshness_guarantee: how to verify and prove fresh seafood"
    
    if any(w in claim_text for w in ["view biển", "view đẹp", "gần biển"]) and any(w in truth_text for w in ["view tường", "không thấy", "xa"]):
        return "honest_location: actual photos, walking time, transport options"
    
    if any(w in claim_text for w in ["phục vụ", "nhiệt tình", "service"]) and any(w in truth_text for w in ["lạnh nhạt", "thái độ", "rude"]):
        return "personal_service: staff training, customer recognition, care"
    
    if any(w in claim_text for w in ["sạch", "clean"]) and any(w in truth_text for w in ["bẩn", "dirty", "dơ"]):
        return "hygiene_transparency: kitchen tours, cleanliness standards"
    
    return "authenticity_content: address this gap honestly with proof"


# ============================================
# 4. CROSS-VALIDATE ENGINE
# ============================================

def cross_validate(node2: Dict, node3: Dict) -> List[Dict]:
    """
    Cross-validate với logging rõ ràng.
    """
    print("\n→ Extracting market claims...")
    claims = extract_market_claims(node2)
    print(f"  Total claims: {len(claims)}")
    for i, c in enumerate(claims[:3], 1):
        print(f"    {i}. {c['claim'][:60]}...")
    
    print("\n→ Extracting customer truths...")
    truths = extract_customer_truths(node3)
    print(f"  Total truths: {len(truths)}")
    for i, t in enumerate(truths[:3], 1):
        print(f"    {i}. [{t['type']}] {t['text'][:60]}...")
    
    results = []
    
    print("\n→ Cross-validating...")
    for ci, claim in enumerate(claims):
        matched = False
        
        for ti, truth in enumerate(truths):
            # Check contradiction
            if is_contradiction(claim["claim"], truth["text"]):
                opp = infer_opportunity(claim, truth)
                results.append({
                    "id": f"m_{len([r for r in results if r.get('mismatch_type') == 'claim_vs_reality'])+1:03d}",
                    "node_2_claim": claim["claim"],
                    "node_2_source": claim["source"],
                    "node_3_truth": truth["text"][:200],
                    "node_3_id": truth["id"],
                    "mismatch_type": "claim_vs_reality",
                    "severity": "high" if truth.get("severity") == "deal_breaker" else "medium",
                    "opportunity": opp,
                    "evidence_ids": [truth["id"]]
                })
                print(f"  ✓ MISMATCH: '{claim['claim'][:40]}...' vs '{truth['text'][:40]}...'")
                matched = True
                break
            
            # Check validation
            elif is_validation(claim["claim"], truth["text"]):
                results.append({
                    "id": f"v_{len([r for r in results if r.get('validation_type') == 'confirmed'])+1:03d}",
                    "node_2_claim": claim["claim"],
                    "node_2_source": claim["source"],
                    "node_3_truth": truth["text"][:200],
                    "node_3_id": truth["id"],
                    "validation_type": "confirmed",
                    "strength": truth.get("type", "neutral"),
                    "action": "amplify_in_content"
                })
                print(f"  ✓ VALIDATION: '{claim['claim'][:40]}...' confirmed by '{truth['text'][:40]}...'")
                matched = True
                break
        
        if not matched:
            print(f"  ✗ No match for: '{claim['claim'][:50]}...'")
    
    return results


# ============================================
# 5. INSIGHT GENERATION
# ============================================

def generate_insights(cross_results: List[Dict], node2: Dict, node3: Dict) -> Dict:
    """
    Generate insights từ cross-validation results.
    """
    mismatches = [r for r in cross_results if r.get("mismatch_type") == "claim_vs_reality"]
    validations = [r for r in cross_results if r.get("validation_type") == "confirmed"]
    
    insights = {
        "real_pain_points": [],
        "market_gaps": [],
        "differentiation_hooks": []
    }
    
    # Pain points từ mismatches
    for m in mismatches:
        insights["real_pain_points"].append({
            "id": m["id"],
            "pain": extract_pain_from_truth(m["node_3_truth"]),
            "description": m["node_3_truth"],
            "evidence": m["evidence_ids"],
            "frequency": "high" if m["severity"] == "high" else "medium",
            "deal_breaker": m["severity"] == "high",
            "content_angle": m["opportunity"]
        })
    
    # Market gaps từ Node 2 content gaps
    for gap in node2.get("market_narrative", {}).get("content_gaps", []):
        insights["market_gaps"].append({
            "gap": gap.get("weakness", "unknown"),
            "description": gap.get("opportunity", ""),
            "competitor_count": gap.get("count", 0),
            "opportunity": f"Create content addressing: {gap.get('weakness', '')}"
        })
    
    # Differentiation hooks
    if mismatches:
        insights["differentiation_hooks"].append({
            "hook": f"Honest about {mismatches[0]['node_2_claim'][:30]}",
            "source": "customer_complaint",
            "application": "transparency_content"
        })
    
    if validations:
        insights["differentiation_hooks"].append({
            "hook": f"Amplify {validations[0]['node_2_claim'][:30]}",
            "source": "customer_praise",
            "application": "testimonial_content"
        })
    
    # Positioning
    insights["positioning_strategy"] = {
        "market_frame": "seafood_restaurant_da_nang",
        "current_narrative": "Competitors promise 'fresh, cheap, beautiful view'",
        "customer_reality": f"Customers experience: {', '.join([p['pain'] for p in insights['real_pain_points'][:2]]) if insights['real_pain_points'] else 'hidden costs and quality issues'}",
        "reframe": "From 'cheapest seafood' to 'honest seafood with transparent pricing'",
        "brand_promise": "We show you the real total cost. No hidden fees. No fake freshness claims.",
        "proof_points": [
            "Real photos with timestamps",
            "Total cost calculator: food + service + extras",
            "Customer reviews with verification"
        ],
        "competitor_weakness_to_attack": [
            m["node_2_claim"] for m in mismatches[:3]
        ]
    }
    
    return insights


def extract_pain_from_truth(truth_text: str) -> str:
    """
    Extract pain point ngắn gọn.
    """
    # Tìm negative context
    patterns = [
        r'nhưng\s+(.{10,50})',
        r'tuy nhiên\s+(.{10,50})',
        r'không\s+(.{5,30})',
        r'thất vọng\s+(.{5,30})',
        r'kém\s+(.{5,30})',
        r'nghi ngờ\s+(.{5,30})',
    ]
    
    for pattern in patterns:
        match = re.search(pattern, truth_text, re.IGNORECASE)
        if match:
            return match.group(1).strip()
    
    # Fallback
    negative_words = ["đắt", "mắc", "tệ", "dở", "lừa", "bẩn", "chậm", "khô", "nhạt", "scam", "fake", "hidden"]
    for word in negative_words:
        idx = truth_text.lower().find(word)
        if idx != -1:
            return truth_text[idx:idx+30].strip()
    
    return truth_text[:30]


# ============================================
# 6. CONTENT BRIEFS
# ============================================

def generate_content_briefs(insights: Dict, node2: Dict, node3: Dict) -> Dict:
    """
    Generate content briefs — template-based.
    """
    pain_points = insights.get("real_pain_points", [])
    positioning = insights.get("positioning_strategy", {})
    
    main_pain = pain_points[0]["pain"] if pain_points else "hidden costs and fake freshness"
    main_angle = pain_points[0]["content_angle"] if pain_points else "transparent pricing"
    
    briefs = {
        "blog_honest_review": {
            "title": f"Ăn Hải Sản Đà Nẵng: {main_pain.title()} (Và Cách Tránh)",
            "hook": f"Story: '{main_pain[:50]}...'",
            "h2_outline": [
                "Cách đọc menu hải sản: giá nào là giá thật?",
                "5 quán hải sản Đà Nẵng: Tổng chi phí thực tế",
                "Dấu hiệu nhận biết hải sản tươi vs đông lạnh",
                "Checklist: 7 câu phải hỏi trước khi gọi món"
            ],
            "key_message": positioning.get("brand_promise", "Transparent pricing, real freshness"),
            "cta": "Lưu checklist — đặt bàn lần sau không bị 'bất ngờ'",
            "tone": "honest_friend",
            "evidence_required": [p["evidence"] for p in pain_points[:3]],
            "target_audience": "family_budget_conscious",
            "seo_keyword_cluster": "hải sản đà nẵng giá rẻ"
        },
        "facebook_ad": {
            "headline": f"Ăn Hải Sản 'Giá Rẻ' = {main_pain[:30]}?",
            "body": f"{main_pain[:80]}. Chúng tôi so sánh TỔNG CHI PHÍ thật.",
            "cta": "Xem so sánh giá thật",
            "audience": "lookalike_family_travelers",
            "differentiation": "Total cost transparency"
        },
        "email_newsletter": {
            "subject": f"Mẹo ăn hải sản Đà Nẵng: 7 câu phải hỏi (trước khi mất tiền)",
            "preview": f"{main_pain[:60]}. Checklist miễn phí.",
            "body_angle": f"Story-driven: 'Lần đầu mình ăn hải sản Đà Nẵng... {main_pain}'",
            "cta": "Tải checklist PDF"
        },
        "tiktok_script": {
            "hook_3s": f"Đặt hải sản 800k/kg? [show bill] {main_pain[:30]}",
            "story_arc": "Order → Arrive → Shock → Compare 5 restaurants → Reveal trick",
            "cta": "Follow để không bị lừa giá nữa"
        },
        "landing_page": {
            "hero_headline": "Hải Sản Đà Nẵng: Giá Thật, Tươi Thật",
            "subheadline": positioning.get("brand_promise", ""),
            "objection_handler": f"Q: Tại sao giá cao hơn quán khác? A: {main_pain[:50]} — chúng tôi không có phí ẩn.",
            "social_proof": "1,200+ khách hàng đã check tổng bill trước khi đặt bàn",
            "cta": "Tính tổng chi phí thực"
        },
        "google_ad": {
            "headline_1": "Hải Sản Đà Nẵng Giá Rẻ?",
            "headline_2": "Xem Tổng Bill Thật Trước",
            "headline_3": "Không Phí Ẩn, Không Lừa",
            "description": f"{main_pain[:80]}. Tính tổng chi phí trước khi đặt bàn.",
            "cta": "So Sánh Giá"
        },
        "instagram_post": {
            "caption_hook": f"Swipe để thấy {main_pain[:40]} 👉",
            "carousel_slides": [
                "Bill quán A: 800k/kg tôm",
                "Bill thực tế: +200k phí + 150k service",
                "Quán chúng tôi: 900k/kg, TỔNG = 900k",
                "Cách tính tổng bill trước khi ăn"
            ],
            "hashtags": "#haisandanang #danangfood #foodtransparency #danangtravel",
            "cta": "Lưu lại để so sánh"
        }
    }
    
    return briefs


# ============================================
# 7. MAIN ENGINE
# ============================================

def node4_engine(node2_file: str = "node2_output.json", node3_file: str = "node3_output.json") -> Dict:
    """
    Node 4 — Insight Fusion.
    """
    print(f"\n{'='*60}")
    print("NODE 4: INSIGHT FUSION (STRATEGIC ENGINE)")
    print(f"{'='*60}")
    
    # Load inputs
    print(f"\n→ Loading Node 2 (Market Narrative)...")
    node2 = load_node_output(node2_file)
    print(f"  Competitors: {node2.get('stats', {}).get('unique_competitors', 0)}")
    
    print(f"\n→ Loading Node 3 (Social Truth)...")
    node3 = load_node_output(node3_file)
    print(f"  Raw voices: {node3.get('stats', {}).get('total_raw_voices', 0)}")
    
    # Cross-validate
    print(f"\n→ Cross-validating Node 2 vs Node 3...")
    cross_results = cross_validate(node2, node3)
    
    mismatches = [r for r in cross_results if r.get("mismatch_type") == "claim_vs_reality"]
    validations = [r for r in cross_results if r.get("validation_type") == "confirmed"]
    
    print(f"\n  Mismatches found: {len(mismatches)}")
    print(f"  Validations found: {len(validations)}")
    
    # Generate insights
    print("\n→ Generating strategic insights...")
    insights = generate_insights(cross_results, node2, node3)
    
    # Generate content briefs
    print("\n→ Generating content briefs...")
    briefs = generate_content_briefs(insights, node2, node3)
    
    # Build output
    output = {
        "input_files": {
            "node2": node2_file,
            "node3": node3_file
        },
        "processing_date": time.strftime("%Y-%m-%d"),
        "stats": {
            "competitors_analyzed": node2.get("stats", {}).get("unique_competitors", 0),
            "raw_voices_analyzed": node3.get("stats", {}).get("total_raw_voices", 0),
            "mismatches_found": len(mismatches),
            "validations_found": len(validations),
            "pain_points": len(insights.get("real_pain_points", [])),
            "market_gaps": len(insights.get("market_gaps", [])),
            "differentiation_hooks": len(insights.get("differentiation_hooks", [])),
            "content_briefs": len(briefs)
        },
        "cross_validation": {
            "mismatches": mismatches,
            "validations": validations
        },
        "insights": insights,
        "content_briefs": briefs,
        "positioning_strategy": insights.get("positioning_strategy", {}),
        "confidence_scores": {
            "pain_points": 0.85 if len(mismatches) > 2 else 0.6,
            "market_gaps": 0.8 if node2.get("market_narrative", {}).get("content_gaps") else 0.5,
            "differentiation_hooks": 0.75 if insights.get("differentiation_hooks") else 0.4,
            "content_angles": 0.9 if briefs else 0.3
        },
        "risk_flags": [
            "LLM not used — template-based generation may lack nuance",
            "Node 3 data may be insufficient for high-confidence insights" if node3.get("stats", {}).get("total_raw_voices", 0) < 10 else None,
            "Cultural nuance may be missed without human review",
        ],
        "human_review_required": [
            "Verify cultural context of pain points",
            "Check legal compliance of competitor naming",
            "Validate pricing claims with actual market data"
        ],
        "next_step": "Feed content_briefs into app content generation engine"
    }
    
    # Clean None
    output["risk_flags"] = [f for f in output["risk_flags"] if f]
    
    # Save
    with open("node4_output.json", "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
    
    print(f"\n{'='*60}")
    print("NODE 4 COMPLETE")
    print(f"Output: node4_output.json")
    print(f"{'='*60}")
    
    # Preview
    print(f"\n--- PREVIEW ---")
    print(f"Competitors: {output['stats']['competitors_analyzed']}")
    print(f"Raw voices: {output['stats']['raw_voices_analyzed']}")
    print(f"Mismatches: {output['stats']['mismatches_found']}")
    print(f"Validations: {output['stats']['validations_found']}")
    print(f"Pain points: {output['stats']['pain_points']}")
    print(f"Content briefs: {output['stats']['content_briefs']}")
    
    if mismatches:
        print(f"\n--- MISMATCHES ---")
        for m in mismatches[:3]:
            print(f"\n[{m['id']}] {m['severity'].upper()}")
            print(f"  Claim: {m['node_2_claim'][:60]}...")
            print(f"  Truth: {m['node_3_truth'][:60]}...")
            print(f"  Opportunity: {m['opportunity']}")
    
    if validations:
        print(f"\n--- VALIDATIONS ---")
        for v in validations[:2]:
            print(f"\n[{v['id']}]")
            print(f"  Claim: {v['node_2_claim'][:60]}...")
            print(f"  Confirmed by: {v['node_3_truth'][:60]}...")
    
    print(f"\n--- POSITIONING ---")
    pos = output.get("positioning_strategy", {})
    print(f"Frame: {pos.get('market_frame', 'N/A')}")
    print(f"Promise: {pos.get('brand_promise', 'N/A')}")
    
    print(f"\n--- SAMPLE BRIEFS ---")
    for content_type, brief in briefs.items():
        print(f"\n[{content_type}]")
        print(f"  Title: {brief.get('title', 'N/A')[:60]}...")
        print(f"  CTA: {brief.get('cta', 'N/A')}")
    
    return output


if __name__ == "__main__":
    node4_engine()


NODE 4: INSIGHT FUSION (STRATEGIC ENGINE)

→ Loading Node 2 (Market Narrative)...
  Competitors: 1

→ Loading Node 3 (Social Truth)...
  Raw voices: 7

→ Cross-validating Node 2 vs Node 3...

→ Extracting market claims...
  Total claims: 3
    1. Market uses article format for seafood content...
    2. Competitor uses content_only monetization...
    3. Dominant content: article...

→ Extracting customer truths...
  Total truths: 7
    1. [complaint] Mình đi ăn ở Mộc Seafood hôm qua, tôm hùm 800k/kg nhưng khôn...
    2. [question] Cho mình hỏi quán hải sản nào ở Đà Nẵng giá rẻ thật, không p...
    3. [complaint] Quán hải sản bên Mỹ Khê nào view đẹp, giá hợp lý không các b...

→ Cross-validating...
  ✗ No match for: 'Market uses article format for seafood content...'
  ✗ No match for: 'Competitor uses content_only monetization...'
  ✗ No match for: 'Dominant content: article...'

  Mismatches found: 0
  Validations found: 0

→ Generating strategic insights...

→ Generating content brie

In [ ]:
import asyncio
import nodriver as uc
import random
import traceback
import re

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"

async def inspect_html_and_click():
    print("🚀 Mở trình duyệt chống chống Bot...")
    browser = await uc.start(headless=False, icon_mode=1)

    try:
        # ========== BƯỚC 1: VÀO GOOGLE SEARCH ==========
        print("🔍 Vào Google tìm kiếm...")
        page = await browser.get("https://www.google.com/search?q=site:facebook.com/mocseafood")

        print("⏳ Chờ trang tải hoàn thiện (6s)...")
        await asyncio.sleep(1)

        # ========== BƯỚC 2: LẤY HTML THẬT TỪ DOM ĐÃ RENDER ==========
        print("📸 Đang lấy toàn bộ nội dung HTML thật từ DOM (live) của trình duyệt...")
        html_content = await page.get_content()

        with open("google_real_inspect.html", "w", encoding="utf-8") as f:
            f.write(html_content)
        print("💾 Đã lưu HTML thật vào file 'google_real_inspect.html' để làm bằng chứng.")

        # ========== BƯỚC 3: TÌM ĐÚNG ELEMENT TRANG CHỦ TRÊN UI ==========
        # Lưu ý: class="zReHs" và jsname="UWckNb" là thuộc tính CHUNG của Google
        # cho MỌI link kết quả tự nhiên (cả link trang chủ và link ảnh đều có).
        # => Không thể dùng riêng class/jsname để phân biệt, phải khóa thêm href tuyệt đối.
        print(f"🎯 Đang tìm chính xác link trang chủ: {TARGET_PAGE_URL}")

        # Ưu tiên 1: selector kết hợp class + jsname + href tuyệt đối (chuẩn nhất)
        selector_strict = f'a.zReHs[jsname="UWckNb"][href="{TARGET_PAGE_URL}"]'
        target_element = await page.select(selector_strict)

        # Ưu tiên 2: nếu Google đổi class/jsname, fallback chỉ dùng href tuyệt đối
        if not target_element:
            print("⚠️ Selector class+jsname không khớp (có thể Google đã đổi class), thử fallback href tuyệt đối...")
            target_element = await page.select(f'a[href="{TARGET_PAGE_URL}"]')

        # Ưu tiên 3: fallback bằng XPath nếu CSS selector vẫn không ăn
        if not target_element:
            print("⚠️ Tìm bằng CSS selector hụt, thử tìm bằng XPath tuyệt đối...")
            result = await page.xpath(f'//a[@href="{TARGET_PAGE_URL}"]')
            if isinstance(result, list) and len(result) > 0:
                target_element = result[0]
            else:
                target_element = result

        if not target_element:
            print("❌ Không tìm thấy link trang chủ chính xác trên trang kết quả.")
            return

        print("✅ Đã tìm thấy đúng element link trang chủ!")
        print(f"   ↳ Tiêu đề nút: {target_element.text.strip()[:50] if target_element.text else 'Trống'}")

        # ========== BƯỚC 4: MÔ PHỎNG VÀ CLICK ==========
        delay = random.uniform(0.5, 2.5)
        print(f"⏳ Nghỉ tay {delay:.2f}s cho mượt...")
        await asyncio.sleep(delay)

        print("🖱️ Thực hiện Click vào link trang chủ...")
        await target_element.click()
        print("  → Đã phát lệnh Click thành công!")

        # ========== BƯỚC 5: CHỜ TRANG ĐÍCH LOAD ==========
        print("⏳ Chờ trang đích điều hướng (10s)...")
        await asyncio.sleep(1)

        print(f"📍 URL hiện tại: {page.url}")
        if "facebook.com/mocseafood" in page.url.lower():
            print("🎉 XUẤT SẮC: Đã vào đúng trang chủ Facebook!")
        else:
            print("⚠️ URL chưa chuyển đổi đúng như mong đợi. Hãy kiểm tra màn hình thực tế.")

    except Exception as e:
        print(f"❌ Đã xảy ra lỗi hệ thống: {e}")
        traceback.print_exc()

    finally:
        print("\n🔒 Đang giữ trình duyệt 5s để quan sát trước khi tắt...")
        await asyncio.sleep(50)
        try:
            browser.stop()
        except Exception:
            pass
        print("🛑 Đã đóng trình duyệt sạch sẽ.")

if __name__ == "__main__":
    await inspect_html_and_click()

In [3]:
import asyncio
import random
import traceback
import nodriver as uc

TARGET_PAGE_URL = "https://www.facebook.com/mocseafood/"


async def inspect_html_and_click():
    browser = None

    try:
        print("🚀 Mở trình duyệt...")

        browser = await uc.start(
            headless=False,
            icon_mode=1
        )

        print("🔍 Vào Google...")

        page = await browser.get(
            "https://www.google.com/search?q=site:facebook.com/mocseafood"
        )

        print("⏳ Đợi kết quả xuất hiện...")

        target_element = None

        for _ in range(30):
            try:
                target_element = await page.select(
                    f'a[href="{TARGET_PAGE_URL}"]'
                )

                if target_element:
                    break

            except Exception:
                pass

            await asyncio.sleep(0.2)

        if not target_element:
            print("❌ Không tìm thấy link.")
            return

        print("✅ Đã tìm thấy link.")

        try:
            await target_element.scroll_into_view()
        except Exception:
            pass

        await asyncio.sleep(
            random.uniform(0.3, 0.8)
        )

        print("🖱️ Click Facebook...")

        await target_element.click()

        await asyncio.sleep(
            random.uniform(2.0, 4.0)
        )

        print("🔎 Kiểm tra tabs...")

        fb_tab = None

        for tab in browser.tabs:
            try:
                url = str(tab.url)

                print("TAB:", url)

                if "facebook.com" in url.lower():
                    fb_tab = tab

            except Exception:
                pass

        if not fb_tab:
            print("❌ Không tìm thấy tab Facebook.")
            return

        print("🎉 Đã tìm thấy Facebook.")

        print("📜 Scroll + Expand Posts + Expand Comments...")
        
        for round_idx in range(10):
        
            print(f"🔄 Round {round_idx + 1}/10")
        
            # =====================================================
            # SCROLL
            # =====================================================
        
            try:
                await fb_tab.evaluate(
                    """
                    window.scrollBy(0, 2500);
                    """
                )
            except Exception:
                pass
        
            await asyncio.sleep(
                random.uniform(1.0, 2.0)
            )
        
            # =====================================================
            # CLICK "SEE MORE"
            # CLICK "VIEW MORE COMMENTS"
            # =====================================================
        
            try:
        
                await fb_tab.evaluate(
                    """
                    (() => {
        
                        const targets = [
                            "See more",
                            "View more comments",
                            "View previous comments",
                            "View more replies",
        
                            "Xem thêm",
                            "Xem thêm bình luận",
                            "Xem phản hồi",
                            "Xem thêm phản hồi"
                        ];
        
                        const all = document.querySelectorAll("*");
        
                        for (const el of all) {
        
                            const text =
                                (el.innerText || "")
                                .trim();
        
                            if (
                                targets.includes(text)
                            ) {
        
                                try {
                                    el.click();
                                } catch(e) {}
        
                            }
                        }
        
                    })();
                    """
                )
        
            except Exception:
                pass
        
            await asyncio.sleep(
                random.uniform(1.0, 2.0)
            )
        
        print("✅ Hoàn tất expand.")
        # ==========================================
        # DEBUG DIALOG
        # ==========================================
        
        try:
        
            dialog_count = await fb_tab.evaluate("""
            document.querySelectorAll('[role="dialog"]').length
            """)
        
            print("FOUND DIALOGS:", dialog_count)
        
            popup_html = await fb_tab.evaluate("""
            (() => {
                const dialogs =
                    document.querySelectorAll('[role="dialog"]');
        
                if (!dialogs.length)
                    return "NO_DIALOG";
        
                return dialogs[dialogs.length - 1].outerHTML;
            })()
            """)
        
            print("POPUP SIZE:", len(popup_html))
        
            with open(
                "popup_only.html",
                "w",
                encoding="utf-8"
            ) as f:
                f.write(popup_html)
        
            print("💾 Saved popup_only.html")
        
        except Exception as e:
        
            print("Popup debug error:", e)
        
        print("📸 Lấy HTML...")
        
        html = await fb_tab.get_content()

        with open(
            "facebook_page.html",
            "w",
            encoding="utf-8"
        ) as f:
            f.write(html)

        print(
            f"💾 Đã lưu facebook_page.html "
            f"({len(html):,} ký tự)"
        )

    except Exception as e:
        print("❌ Lỗi:", e)
        traceback.print_exc()

    finally:
        print("⏳ Giữ trình duyệt 30s...")
        await asyncio.sleep(30)

        try:
            if browser:
                browser.stop()
        except Exception:
            pass
            
if __name__ == "__main__":
    await inspect_html_and_click()

🚀 Mở trình duyệt...
🔍 Vào Google...
⏳ Đợi kết quả xuất hiện...
✅ Đã tìm thấy link.
🖱️ Click Facebook...
🔎 Kiểm tra tabs...
TAB: https://www.google.com/search?q=site:facebook.com/mocseafood&sei=iP0zasO3IsWXvr0Pncy92QM
🎉 Đã tìm thấy Facebook.
📜 Scroll + Expand Posts + Expand Comments...
🔄 Round 1/10
🔄 Round 2/10
🔄 Round 3/10
🔄 Round 4/10
🔄 Round 5/10
🔄 Round 6/10
🔄 Round 7/10
🔄 Round 8/10
🔄 Round 9/10
🔄 Round 10/10
✅ Hoàn tất expand.
FOUND DIALOGS: 2
POPUP SIZE: 970053
💾 Saved popup_only.html
📸 Lấy HTML...
💾 Đã lưu facebook_page.html (3,013,559 ký tự)
⏳ Giữ trình duyệt 30s...


In [5]:
# trích xuất json crew


from bs4 import BeautifulSoup
import re
import json

# ============================================================
# CONFIG
# ============================================================

INPUT_FILE = "popup_only.html"
OUTPUT_FILE = "brand_raw.json"

# ============================================================
# LOAD HTML
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

raw_text = soup.get_text(
    "\n",
    strip=True
)

# ============================================================
# CLEAN FACEBOOK NOISE
# ============================================================

NOISE_LINES = {
    "Like",
    "Comment",
    "Share",
    "See more",
    "View more comments",
    "All reactions:",
    "Online status indicator",
    "Active Status indicator",
    "Privacy",
    "Terms",
    "Advertising",
    "Ad choices",
    "Cookies",
    "More",
    "Active",
    "Log in",
    "Forgotten account?"
}

clean_lines = []

for line in raw_text.split("\n"):

    line = line.strip()

    if not line:
        continue

    if line in NOISE_LINES:
        continue

    # reaction count
    if re.fullmatch(r"\d+", line):
        continue

    # video time
    if re.fullmatch(r"\d+:\d+", line):
        continue

    clean_lines.append(line)

clean_text = "\n".join(clean_lines)

# ============================================================
# PAGE INFO
# ============================================================

page_info = {}

title_tag = soup.find("title")

if title_tag:
    page_info["title"] = title_tag.get_text(strip=True)

m = re.search(
    r'(\d+(?:\.\d+)?K)\s+followers',
    clean_text,
    re.IGNORECASE
)

if m:
    page_info["followers"] = m.group(1)

m = re.search(
    r'(\d+(?:\.\d+)?K)\s+following',
    clean_text,
    re.IGNORECASE
)

if m:
    page_info["following"] = m.group(1)

# ============================================================
# INTRO SECTION
# ============================================================

intro_section = ""

try:

    start = clean_text.index("Intro")

    possible_ends = []

    for marker in [
        "\nPhotos",
        "\nPosts",
        "\nAbout",
        "\nPrivacy"
    ]:

        pos = clean_text.find(marker, start + 5)

        if pos != -1:
            possible_ends.append(pos)

    if possible_ends:
        end = min(possible_ends)
        intro_section = clean_text[start:end]

except ValueError:
    pass

# ============================================================
# STRUCTURED DATA
# ============================================================

# phones

raw_phones = re.findall(
    r'(?:\+84\s?)?(?:0\d{2,3}[\s.-]?\d{3}[\s.-]?\d{3,4})',
    clean_text
)

phones = set()

for phone in raw_phones:

    phone = re.sub(
        r"[\s.-]",
        "",
        phone
    )

    if phone.startswith("+84"):
        phone = "0" + phone[3:]

    phones.add(phone)

phones = sorted(list(phones))

# emails

emails = sorted(set(
    re.findall(
        r'[\w\.-]+@[\w\.-]+\.\w+',
        clean_text
    )
))

# domains

domains = sorted(set(
    re.findall(
        r'[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        clean_text
    )
))

# ============================================================
# CONTACT BLOCKS
# ============================================================

contact_blocks = []

intro_lines = intro_section.split("\n")

for i, line in enumerate(intro_lines):

    if any(
        k.lower() in line.lower()
        for k in [
            "cơ sở",
            "hotline",
            "đà nẵng",
            "nha trang",
            "@",
            ".com",
            "restaurant",
            "delivery",
            "takeaway"
        ]
    ):
        start = max(0, i - 2)
        end = min(len(intro_lines), i + 3)

        block = "\n".join(
            intro_lines[start:end]
        )

        contact_blocks.append(block)

contact_blocks = list(dict.fromkeys(contact_blocks))

# ============================================================
# POSTS
# ============================================================

posts = []

chunks = clean_text.split(
    "Shared with Public"
)

for chunk in chunks:

    chunk = chunk.strip()

    if len(chunk) < 150:
        continue

    if "followers" in chunk:
        continue

    posts.append(chunk)

posts = posts[:10]

# ============================================================
# RESULT
# ============================================================

result = {
    "page_info": page_info,

    "sections": {
        "intro": intro_section
    },

    "structured": {
        "phones": phones,
        "emails": emails,
        "domains": domains
    },

    "semi_structured": {
        "contact_blocks": contact_blocks,
        "post_blocks": posts
    },

    "raw_text": clean_text
}

# ============================================================
# SAVE
# ============================================================

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        result,
        f,
        ensure_ascii=False,
        indent=2
    )

# ============================================================
# PREVIEW
# ============================================================

print("=" * 80)
print("PAGE INFO")
print("=" * 80)

print(json.dumps(
    page_info,
    ensure_ascii=False,
    indent=2
))

print("\n")
print("=" * 80)
print("INTRO SECTION")
print("=" * 80)

print(intro_section[:3000])

print("\n")
print("=" * 80)
print("PHONES")
print("=" * 80)

print(phones)

print("\n")
print("=" * 80)
print("EMAILS")
print("=" * 80)

print(emails)

print("\n")
print("=" * 80)
print("DOMAINS")
print("=" * 80)

print(domains)

print("\n")
print("=" * 80)
print("CONTACT BLOCKS")
print("=" * 80)

for i, block in enumerate(contact_blocks, 1):
    print(f"\n--- BLOCK {i} ---")
    print(block)

print("\n")
print("=" * 80)
print("POST COUNT")
print("=" * 80)

print(len(posts))

if posts:
    print("\nFIRST POST PREVIEW\n")
    print(posts[0][:1500])

print("\n")
print("=" * 80)
print("RAW TEXT SIZE")
print("=" * 80)

print(len(clean_text))

print(f"\n💾 Saved -> {OUTPUT_FILE}")

PAGE INFO
{
  "title": "Shared with Public"
}


INTRO SECTION



PHONES
['0888622662', '0905665058', '0934981180', '0964531153']


EMAILS
[]


DOMAINS
[]


CONTACT BLOCKS


POST COUNT
10

FIRST POST PREVIEW

THƯỞNG THỨC HẢI SẢN CHUẨN VỊ - SĂN TRỌN VIEW PHÁO HOA DIFF 2026 TẠI MỘC ĐÀ NẴNG!
Lễ hội Pháo hoa Quốc tế Đà Nẵng (DIFF) đang làm nóng rực bầu không khí thành phố! Bạn muốn chiêm ngưỡng những màn trình diễn ánh sáng mãn nhãn nhưng e ngại cảnh chen chúc? Hãy để Mộc Seafood mang đến cho bạn một đêm tiệc hoàn hảo:
Tọa độ ngắm pháo hoa cực "chill": Không gian thoáng đãng cùng tầm nhìn lý tưởng, giúp bạn thoải mái tận hưởng bầu trời rực rỡ mà không cần chen lấn.
Bàn tiệc hải sản đỉnh cao: Nâng ly cùng người thân, nhâm nhi hải sản đậm đà vị biển và thu trọn vào tầm mắt khung cảnh pháo hoa rực rỡ.
Inbox Fanpage hoặc gọi Hotline ngay để đặt bàn trước và giữ vị trí đẹp nhất!
SAVOR AUTHENTIC SEAFOOD - CATCH THE PERFECT VIEW OF DIFF 2026 AT MỘC DA NANG!
DIFF 2026 is heating up the city! Want t